# Aprimoramento de Small Language Models (SLMs)

Este notebook apresenta um experimento científico para aprimoramento de SLMs, combinando Knowledge Distillation e Prompt Engineering. O objetivo é comparar técnicas de destilação e engenharia de prompt em ambiente computacional limitado.

In [ ]:
from huggingface_hub import login

# Log in to Hugging Face
login()

In [24]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
# -*- coding: utf-8 -*-
"""
Projeto Científico: Aprimoramento de Small Language Models (SLMs)
Combinação de Knowledge Distillation e Prompt Engineering

Data: 2025
Objetivo: Comparar técnicas de destilação e engenharia de prompt em ambiente limitado
"""

'\nProjeto Científico: Aprimoramento de Small Language Models (SLMs)\nCombinação de Knowledge Distillation e Prompt Engineering\n\nData: 2025\nObjetivo: Comparar técnicas de destilação e engenharia de prompt em ambiente limitado\n'

In [5]:
!pip install torch torchvision torchaudio transformers datasets tokenizers evaluate accelerate peft numpy pandas scipy scikit-learn matplotlib seaborn plotly tqdm nltk spacy psutil rich python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 89.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 94.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 61.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.7/188.7 MB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 86.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.2 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12

## Importação de bibliotecas
Importação dos principais pacotes utilizados para machine learning, manipulação de dados e visualização.

In [6]:
import os
import json
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from collections import Counter, defaultdict
import warnings
warnings.filterwarnings('ignore')

## Imports para Machine Learning e Transformers
Carregamento de datasets, métricas e modelos pré-treinados.

In [7]:
# Imports para ML
from datasets import load_dataset, Dataset
from evaluate import load
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    TrainingArguments, Trainer, DataCollatorForLanguageModeling
)
from peft import get_peft_model, LoraConfig, TaskType, PeftModel
from torch.optim import AdamW
from tqdm.auto import tqdm
import re

## Imports adicionais para técnicas avançadas
Importação condicional de pacotes para RAG e análise de similaridade.

In [8]:
# Imports adicionais para técnicas avançadas
try:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import cosine_similarity
except ImportError:
    print("⚠️ Aviso: sklearn não disponível - algumas funcionalidades RAG limitadas")
    TfidfVectorizer = None
    cosine_similarity = None

## Configuração de seeds para reprodutibilidade
Definição de seeds para garantir resultados reprodutíveis.

In [9]:
# Configuração de seeds para reprodutibilidade
torch.manual_seed(42)
np.random.seed(42)

## Definição das classes principais
As classes a seguir organizam o experimento em módulos: configuração, logging, gerenciamento de dados, distilação, engenharia de prompt, avaliação, geração de relatório e execução principal.

###Classe ExperimentConfig
Configuração de parâmetros e modelos

In [10]:
class ExperimentConfig:
    """Configuração centralizada do experimento científico"""

    # Modelos
    STUDENT_MODEL_ID = "google/gemma-2b"
    TEACHER_MODEL_ID = "microsoft/DialoGPT-medium"  # Modelo mais acessível que Mistral-7B

    # Hiperparâmetros KD
    TEMPERATURE = 3.0  # Aumentado para melhor transferência
    ALPHA = 0.7
    LEARNING_RATE = 3e-4
    NUM_EPOCHS = 2
    BATCH_SIZE = 2
    MAX_LENGTH = 512

    # Configuração do ambiente
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    EVAL_SUBSET_SIZE = 100  # Para ambiente limitado
    KD_TRANSFER_SIZE = 1000  # Dataset reduzido

    # LoRA Config
    LORA_R = 8  # Reduzido para economia de memória
    LORA_ALPHA = 16
    LORA_DROPOUT = 0.1

    # Paths
    OUTPUT_DIR = "./scientific_results"
    MODELS_DIR = f"{OUTPUT_DIR}/models"
    REPORTS_DIR = f"{OUTPUT_DIR}/reports"
    FIGURES_DIR = f"{OUTPUT_DIR}/figures"

###ScientificLogger
Sistema de registro das fases do experimento científico. Armazena timestamp, configurações, métricas e resultados em estrutura JSON. Também cria diretórios para organizar saídas e permite salvar todo o histórico do experimento para fins de reprodutibilidade e auditoria.

In [11]:
class ScientificLogger:
    """Sistema de logging para experimentos científicos"""

    def __init__(self, config):
        self.config = config
        self.experiment_log = {
            'timestamp': datetime.now().isoformat(),
            'config': vars(config),
            'phases': {},
            'results': {},
            'metrics': {}
        }

        # Criar diretórios
        os.makedirs(config.OUTPUT_DIR, exist_ok=True)
        os.makedirs(config.MODELS_DIR, exist_ok=True)
        os.makedirs(config.REPORTS_DIR, exist_ok=True)
        os.makedirs(config.FIGURES_DIR, exist_ok=True)

    def log_phase(self, phase_name, data):
        """Log de uma fase do experimento"""
        self.experiment_log['phases'][phase_name] = {
            'timestamp': datetime.now().isoformat(),
            'data': data
        }
        print(f"[{datetime.now().strftime('%H:%M:%S')}] FASE {phase_name}: {data.get('status', 'Concluída')}")

    def save_experiment_log(self):
        """Salva o log completo do experimento"""
        with open(f"{self.config.REPORTS_DIR}/experiment_log.json", 'w') as f:
            json.dump(self.experiment_log, f, indent=2)


### `DatasetManager`  
Carrega o dataset SQuAD, gera estatísticas descritivas e cria subconjuntos para avaliação, distilação e exemplos ICL.

In [12]:
class DatasetManager:
    """Gerenciador científico de datasets com estatísticas"""

    def __init__(self, config, logger):
        self.config = config
        self.logger = logger

    def load_and_analyze_data(self):
        """Carrega e analisa os dados de forma científica"""
        print("Carregando e analisando datasets...")

        # Carregar SQuAD
        squad_val = load_dataset("squad", split="validation")
        squad_train = load_dataset("squad", split="train")
        squad_metric = load("squad")

        # Estatísticas descritivas
        val_contexts_len = [len(ex['context'].split()) for ex in squad_val.select(range(1000))]
        val_questions_len = [len(ex['question'].split()) for ex in squad_val.select(range(1000))]

        stats = {
            'validation_size': len(squad_val),
            'train_size': len(squad_train),
            'avg_context_length': np.mean(val_contexts_len),
            'avg_question_length': np.mean(val_questions_len),
            'std_context_length': np.std(val_contexts_len),
            'std_question_length': np.std(val_questions_len)
        }

        # Criar subsets para experimento
        kd_transfer_set = squad_train.shuffle(seed=42).select(range(self.config.KD_TRANSFER_SIZE))
        eval_subset = squad_val.shuffle(seed=42).select(range(self.config.EVAL_SUBSET_SIZE))
        icl_examples = squad_train.select(range(5))

        datasets = {
            'evaluation': eval_subset,
            'kd_transfer': kd_transfer_set,
            'icl_examples': icl_examples,
            'metrics_calculator': squad_metric
        }

        self.logger.log_phase("DATASET_ANALYSIS", {'stats': stats})
        print(f"✅ Datasets preparados - Eval: {len(eval_subset)}, KD: {len(kd_transfer_set)}")

        return datasets, stats

### `ScientificLogger`  
Registra fases do experimento, cria diretórios de saída e salva os logs em JSON para rastreabilidade e reprodutibilidade.

In [13]:
class ScientificLogger:
    """Sistema de logging para experimentos científicos"""

    def __init__(self, config):
        self.config = config
        self.experiment_log = {
            'timestamp': datetime.now().isoformat(),
            'config': vars(config),
            'phases': {},
            'results': {},
            'metrics': {}
        }

        # Criar diretórios
        os.makedirs(config.OUTPUT_DIR, exist_ok=True)
        os.makedirs(config.MODELS_DIR, exist_ok=True)
        os.makedirs(config.REPORTS_DIR, exist_ok=True)
        os.makedirs(config.FIGURES_DIR, exist_ok=True)

    def log_phase(self, phase_name, data):
        """Log de uma fase do experimento"""
        self.experiment_log['phases'][phase_name] = {
            'timestamp': datetime.now().isoformat(),
            'data': data
        }
        print(f"[{datetime.now().strftime('%H:%M:%S')}] FASE {phase_name}: {data.get('status', 'Concluída')}")

    def save_experiment_log(self):
        """Salva o log completo do experimento"""
        with open(f"{self.config.REPORTS_DIR}/experiment_log.json", 'w') as f:
            json.dump(self.experiment_log, f, indent=2)


### `DatasetManager`  
Gerencia datasets científicos: carrega SQuAD, calcula estatísticas descritivas, cria subsets para experimentos e registra as fases do processamento para análise e rastreabilidade.


In [14]:
class DatasetManager:
    """Gerenciador científico de datasets com estatísticas"""

    def __init__(self, config, logger):
        self.config = config
        self.logger = logger

    def load_and_analyze_data(self):
        """Carrega e analisa os dados de forma científica"""
        print("Carregando e analisando datasets...")

        # Carregar SQuAD
        squad_val = load_dataset("squad", split="validation")
        squad_train = load_dataset("squad", split="train")
        squad_metric = load("squad")

        # Estatísticas descritivas
        val_contexts_len = [len(ex['context'].split()) for ex in squad_val.select(range(1000))]
        val_questions_len = [len(ex['question'].split()) for ex in squad_val.select(range(1000))]

        stats = {
            'validation_size': len(squad_val),
            'train_size': len(squad_train),
            'avg_context_length': np.mean(val_contexts_len),
            'avg_question_length': np.mean(val_questions_len),
            'std_context_length': np.std(val_contexts_len),
            'std_question_length': np.std(val_questions_len)
        }

        # Criar subsets para experimento
        kd_transfer_set = squad_train.shuffle(seed=42).select(range(self.config.KD_TRANSFER_SIZE))
        eval_subset = squad_val.shuffle(seed=42).select(range(self.config.EVAL_SUBSET_SIZE))
        icl_examples = squad_train.select(range(5))

        datasets = {
            'evaluation': eval_subset,
            'kd_transfer': kd_transfer_set,
            'icl_examples': icl_examples,
            'metrics_calculator': squad_metric
        }

        self.logger.log_phase("DATASET_ANALYSIS", {'stats': stats})
        print(f"✅ Datasets preparados - Eval: {len(eval_subset)}, KD: {len(kd_transfer_set)}")

        return datasets, stats

### `KnowledgeDistiller`  
Implementa distilação de conhecimento científica, combinando perdas KD e Cross Entropy. Suporta configuração de modelos com LoRA para eficiência, treinamento com logits do professor, auto-distilação (sem professor externo) e logging detalhado das métricas e fases do processo.

In [15]:
class KnowledgeDistiller:
    """Implementação científica de Knowledge Distillation"""

    def __init__(self, config, logger):
        self.config = config
        self.logger = logger

    def compute_kd_loss(self, student_logits, teacher_logits, labels=None):
        """
        Computa loss de KD com opção de combinar com CE loss

        Args:
            student_logits: Logits do modelo estudante
            teacher_logits: Logits do modelo professor
            labels: Labels verdadeiros (opcional para CE loss)
        """
        # Soft targets do professor
        teacher_probs = F.softmax(teacher_logits / self.config.TEMPERATURE, dim=-1)
        student_log_probs = F.log_softmax(student_logits / self.config.TEMPERATURE, dim=-1)

        # KL Divergence Loss
        kd_loss = F.kl_div(student_log_probs, teacher_probs, reduction='batchmean')
        kd_loss *= (self.config.TEMPERATURE ** 2)

        # Combinar com Cross Entropy se labels disponíveis
        if labels is not None:
            ce_loss = F.cross_entropy(student_logits.view(-1, student_logits.size(-1)),
                                    labels.view(-1), ignore_index=-100)
            total_loss = self.config.ALPHA * kd_loss + (1 - self.config.ALPHA) * ce_loss
            return total_loss, kd_loss.item(), ce_loss.item()

        return kd_loss, kd_loss.item(), 0.0

    def setup_lora_model(self, base_model, target_modules=None):
        """Configura modelo com LoRA para eficiência"""
        if target_modules is None:
            target_modules = ["q_proj", "v_proj", "k_proj", "o_proj"]

        lora_config = LoraConfig(
            r=self.config.LORA_R,
            lora_alpha=self.config.LORA_ALPHA,
            target_modules=target_modules,
            lora_dropout=self.config.LORA_DROPOUT,
            bias="none",
            task_type=TaskType.CAUSAL_LM
        )

        peft_model = get_peft_model(base_model, lora_config)
        peft_model.print_trainable_parameters()
        return peft_model

    def distill_with_logits(self, student_model, teacher_model,
                            student_tokenizer, teacher_tokenizer,
                            kd_dataset, technique_name):
        """
        Destilação baseada em logits com handling robusto de erros
        """
        print(f"Iniciando {technique_name}...")

        try:
            # Setup LoRA no estudante
            student_peft = self.setup_lora_model(student_model)
            optimizer = AdamW(student_peft.parameters(), lr=self.config.LEARNING_RATE)

            # Freeze teacher
            teacher_model.eval()
            for param in teacher_model.parameters():
                param.requires_grad = False

            # Métricas de treinamento
            training_metrics = {
                'epochs': [],
                'kd_losses': [],
                'ce_losses': [],
                'total_losses': []
            }

            # Verificar compatibilidade de vocabulários antes do loop
            sample_text = "teste"
            teacher_test = teacher_tokenizer(sample_text, return_tensors="pt")
            student_test = student_tokenizer(sample_text, return_tensors="pt")

            with torch.no_grad():
                teacher_test_out = teacher_model(**teacher_test.to(self.config.DEVICE))
                student_test_out = student_peft(**student_test.to(self.config.DEVICE))

            vocab_compatible = (teacher_test_out.logits.shape[-1] ==
                              student_test_out.logits.shape[-1])

            if not vocab_compatible:
                print("⚠️ Vocabulários incompatíveis - usando CE loss apenas")

            for epoch in range(self.config.NUM_EPOCHS):
                epoch_kd_losses = []
                epoch_ce_losses = []
                epoch_total_losses = []

                student_peft.train()

                # Sample do dataset para cada época
                epoch_size = min(200, len(kd_dataset))
                epoch_data = kd_dataset.shuffle(seed=42+epoch).select(range(epoch_size))

                for step, batch in enumerate(tqdm(epoch_data,
                                                desc=f"Época {epoch+1}/{self.config.NUM_EPOCHS}")):
                    try:
                        # Criar prompt estruturado
                        if isinstance(batch['answers']['text'], list) and len(batch['answers']['text']) > 0:
                            answer = batch['answers']['text'][0]
                        else:
                            answer = "N/A"

                        prompt = f"Contexto: {batch['context'][:500]}\n\nPergunta: {batch['question']}\n\nResposta: {answer}"

                        # Tokenização com handling de erros
                        try:
                            teacher_inputs = teacher_tokenizer(
                                prompt,
                                return_tensors="pt",
                                max_length=self.config.MAX_LENGTH,
                                truncation=True,
                                padding=True
                            ).to(self.config.DEVICE)

                            student_inputs = student_tokenizer(
                                prompt,
                                return_tensors="pt",
                                max_length=self.config.MAX_LENGTH,
                                truncation=True,
                                padding=True
                            ).to(self.config.DEVICE)
                        except Exception as e:
                            print(f"Erro na tokenização: {e}")
                            continue

                        # Forward pass com handling de erros
                        try:
                            with torch.no_grad():
                                teacher_outputs = teacher_model(**teacher_inputs)

                            student_outputs = student_peft(**student_inputs,
                                                        labels=student_inputs['input_ids'])
                        except Exception as e:
                            print(f"Erro no forward pass: {e}")
                            continue

                        # Compute loss
                        if vocab_compatible:
                            try:
                                loss, kd_loss_val, ce_loss_val = self.compute_kd_loss(
                                    student_outputs.logits,
                                    teacher_outputs.logits,
                                    student_inputs['input_ids']
                                )
                            except Exception as e:
                                print(f"Erro no cálculo KD loss: {e}")
                                loss = student_outputs.loss
                                kd_loss_val = 0.0
                                ce_loss_val = loss.item()
                        else:
                            # Fallback para CE loss apenas
                            loss = student_outputs.loss
                            kd_loss_val = 0.0
                            ce_loss_val = loss.item()

                        # Backward pass com gradient clipping
                        try:
                            loss.backward()
                            torch.nn.utils.clip_grad_norm_(student_peft.parameters(), 1.0)
                            optimizer.step()
                            optimizer.zero_grad()
                        except Exception as e:
                            print(f"Erro no backward pass: {e}")
                            optimizer.zero_grad()
                            continue

                        # Coletar métricas
                        epoch_kd_losses.append(kd_loss_val)
                        epoch_ce_losses.append(ce_loss_val)
                        epoch_total_losses.append(loss.item())

                        if step % 50 == 0 and step > 0:
                            print(f"Step {step}: KD Loss: {kd_loss_val:.4f}, CE Loss: {ce_loss_val:.4f}")

                    except Exception as e:
                        print(f"Erro no step {step}: {e}")
                        continue

                # Salvar métricas da época se houver dados válidos
                if epoch_total_losses:
                    training_metrics['epochs'].append(epoch + 1)
                    training_metrics['kd_losses'].append(np.mean(epoch_kd_losses))
                    training_metrics['ce_losses'].append(np.mean(epoch_ce_losses))
                    training_metrics['total_losses'].append(np.mean(epoch_total_losses))

                    print(f"Época {epoch+1} concluída - Loss médio: {np.mean(epoch_total_losses):.4f}")
                else:
                    print(f"⚠️ Época {epoch+1} sem dados válidos")

            # Salvar modelo se treinamento foi bem-sucedido
            if training_metrics['epochs']:
                output_path = f"{self.config.MODELS_DIR}/{technique_name.lower().replace(' ', '_')}"
                try:
                    os.makedirs(output_path, exist_ok=True)
                    student_peft.save_pretrained(output_path)
                    student_tokenizer.save_pretrained(output_path)
                    print(f"✅ Modelo salvo em: {output_path}")
                except Exception as e:
                    print(f"⚠️ Erro ao salvar modelo: {e}")

                # Log da técnica
                self.logger.log_phase(f"KD_{technique_name}", {
                    'status': 'Concluída',
                    'output_path': output_path,
                    'training_metrics': training_metrics,
                    'final_total_loss': training_metrics['total_losses'][-1] if training_metrics['total_losses'] else 0,
                    'vocab_compatible': vocab_compatible
                })
            else:
                print(f"❌ Treinamento {technique_name} falhou - sem métricas válidas")
                training_metrics = {
                    'epochs': [0],
                    'kd_losses': [0],
                    'ce_losses': [0],
                    'total_losses': [0]
                }

            return student_peft, training_metrics

        except Exception as e:
            print(f"❌ Erro crítico em {technique_name}: {e}")
            return student_model, {
                'epochs': [0],
                'kd_losses': [0],
                'ce_losses': [0],
                'total_losses': [0],
                'error': str(e)
            }

    def self_distillation(self, base_model, tokenizer, kd_dataset):
        """Implementa auto-destilação sem professor externo"""
        print("🔄 Executando Auto-Destilação...")

        # Fase 1: Fine-tune padrão para criar "professor interno"
        specialist_model = self.setup_lora_model(base_model)

        # Setup para fine-tuning padrão
        def tokenize_function(examples):
            prompts = [f"Contexto: {ctx}\n\nPergunta: {q}\n\nResposta: {ans['text'][0]}"
                      for ctx, q, ans in zip(examples['context'], examples['question'], examples['answers'])]
            return tokenizer(prompts, truncation=True, max_length=self.config.MAX_LENGTH, padding=True)

        tokenized_dataset = kd_dataset.select(range(500)).map(tokenize_function, batched=True)

        training_args = TrainingArguments(
            output_dir=f"{self.config.MODELS_DIR}/temp_specialist",
            num_train_epochs=1,
            per_device_train_batch_size=2,
            learning_rate=2e-5,
            logging_steps=50,
            save_strategy="no",
            report_to=None
        )

        trainer = Trainer(
            model=specialist_model,
            args=training_args,
            train_dataset=tokenized_dataset,
            tokenizer=tokenizer,
            data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
        )

        print("Fase 1: Criando professor especialista...")
        trainer.train()

        # Fase 2: Auto-destilação
        print("Fase 2: Auto-destilação...")
        new_student = AutoModelForCausalLM.from_pretrained(self.config.STUDENT_MODEL_ID)

        return self.distill_with_logits(
            new_student, specialist_model, tokenizer, tokenizer,
            kd_dataset, "Auto_Distillation"
        )

### `PromptEngineer`  
Sistema científico para engenharia de prompts, incluindo estratégias zero-shot, few-shot e Chain-of-Thought (CoT). Suporta avaliação detalhada com métricas, auto-consistência via múltiplos caminhos amostrados, extração inteligente de respostas e avaliação avançada usando workflow multiagente.


In [16]:
class PromptEngineer:
    """Sistema científico de engenharia de prompts"""

    def __init__(self, config, logger):
        self.config = config
        self.logger = logger

    def evaluate_with_prompt_strategy(self, model, tokenizer, eval_dataset,
                                    metrics_calc, strategy_name, prompt_func, **kwargs):
        """
        Avaliação científica com estratégia de prompt específica
        """
        print(f"🔍 Avaliando: {strategy_name}")

        model.eval()
        predictions = []
        references = []
        response_lengths = []
        generation_times = []

        for idx, example in enumerate(tqdm(eval_dataset, desc=strategy_name)):
            # Aplicar estratégia de prompt
            prompt = prompt_func(example, **kwargs)

            # Tokenizar
            inputs = tokenizer(
                prompt,
                return_tensors="pt",
                max_length=self.config.MAX_LENGTH,
                truncation=True
            ).to(self.config.DEVICE)

            # Medir tempo de geração
            start_time = torch.cuda.Event(enable_timing=True) if torch.cuda.is_available() else None
            end_time = torch.cuda.Event(enable_timing=True) if torch.cuda.is_available() else None

            if start_time:
                start_time.record()

            # Gerar resposta
            with torch.no_grad():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=60,
                    pad_token_id=tokenizer.eos_token_id,
                    temperature=0.7 if 'consistency' in strategy_name.lower() else 1.0,
                    do_sample='consistency' in strategy_name.lower()
                )

            if start_time:
                end_time.record()
                torch.cuda.synchronize()
                gen_time = start_time.elapsed_time(end_time)
                generation_times.append(gen_time)

            # Extrair resposta
            full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
            predicted_answer = full_text[len(prompt):].strip()

            # Post-processamento específico da estratégia
            if "cot" in strategy_name.lower():
                predicted_answer = self._extract_final_answer(predicted_answer)

            response_lengths.append(len(predicted_answer.split()))

            predictions.append({
                'id': example['id'],
                'prediction_text': predicted_answer
            })
            references.append({
                'id': example['id'],
                'answers': example['answers']
            })

        # Calcular métricas
        results = metrics_calc.compute(predictions=predictions, references=references)

        # Adicionar métricas de eficiência
        results.update({
            'avg_response_length': np.mean(response_lengths),
            'std_response_length': np.std(response_lengths),
            'avg_generation_time_ms': np.mean(generation_times) if generation_times else 0,
            'total_examples': len(predictions)
        })

        return results

    def _extract_final_answer(self, text):
        """Extrai resposta final de texto CoT"""
        patterns = [
            r'(?:a resposta (?:final )?é|resposta):?\s*(.+?)(?:\n|$)',
            r'(?:portanto|assim|logo),?\s*(.+?)(?:\n|$)',
            r'(?:^|\n)([^.\n]+)(?:\.|$)'  # Última frase
        ]

        for pattern in patterns:
            match = re.search(pattern, text, re.IGNORECASE | re.MULTILINE)
            if match:
                return match.group(1).strip()

        # Fallback: última linha não vazia
        lines = [line.strip() for line in text.split('\n') if line.strip()]
        return lines[-1] if lines else text.strip()

    def zero_shot_prompt(self, example):
        """Prompt zero-shot básico"""
        return f"Contexto: {example['context']}\n\nPergunta: {example['question']}\n\nResposta:"

    def few_shot_prompt(self, example, icl_examples=None, num_shots=3):
        """Prompt few-shot com exemplos"""
        if not icl_examples:
            return self.zero_shot_prompt(example)

        # Construir prefixo com exemplos
        prefix = ""
        for ex in icl_examples.select(range(min(num_shots, len(icl_examples)))):
            answer = ex['answers']['text'][0] if ex['answers']['text'] else "N/A"
            prefix += f"Contexto: {ex['context']}\n\nPergunta: {ex['question']}\n\nResposta: {answer}\n\n---\n\n"

        return prefix + self.zero_shot_prompt(example)

    def cot_prompt(self, example):
        """Prompt Chain-of-Thought"""
        return (f"Contexto: {example['context']}\n\n"
                f"Pergunta: {example['question']}\n\n"
                f"Vamos pensar passo a passo para responder esta pergunta:\n")

    def self_consistency_evaluation(self, model, tokenizer, eval_dataset,
                                  metrics_calc, num_paths=5):
        """
        Avaliação com auto-consistência científica
        """
        print(f"🎯 Auto-Consistência com {num_paths} caminhos...")

        predictions = []
        references = []
        consistency_scores = []

        for example in tqdm(eval_dataset.select(range(50)), desc="Self-Consistency"):  # Subset menor para eficiência
            prompt = self.cot_prompt(example)
            inputs = tokenizer(prompt, return_tensors="pt", max_length=self.config.MAX_LENGTH, truncation=True).to(self.config.DEVICE)

            # Gerar múltiplos caminhos
            path_answers = []
            for _ in range(num_paths):
                with torch.no_grad():
                    outputs = model.generate(
                        **inputs,
                        max_new_tokens=100,
                        do_sample=True,
                        temperature=0.7,
                        top_k=50,
                        pad_token_id=tokenizer.eos_token_id
                    )

                completion = tokenizer.decode(outputs[0], skip_special_tokens=True)[len(prompt):].strip()
                final_answer = self._extract_final_answer(completion)
                if final_answer:
                    path_answers.append(final_answer)

            # Calcular consistência e resposta majoritária
            if path_answers:
                answer_counts = Counter(path_answers)
                most_common_answer, most_common_count = answer_counts.most_common(1)[0]
                consistency_score = most_common_count / len(path_answers)
            else:
                most_common_answer = ""
                consistency_score = 0.0

            consistency_scores.append(consistency_score)
            predictions.append({'id': example['id'], 'prediction_text': most_common_answer})
            references.append({'id': example['id'], 'answers': example['answers']})

        results = metrics_calc.compute(predictions=predictions, references=references)
        results['avg_consistency'] = np.mean(consistency_scores)
        results['std_consistency'] = np.std(consistency_scores)

        return results

    def agentic_workflow_evaluation(self, model, tokenizer, eval_dataset, metrics_calc):
        """Avaliação científica usando o fluxo de trabalho com agentes."""
        strategy_name = "Workflow Agente-Executor"
        print(f"🔍 Avaliando: {strategy_name}")

        model.eval()
        predictions = []
        references = []

        for example in tqdm(eval_dataset, desc=strategy_name):
            try:
                # --- Passo 1: Agente Analista ---
                analyst_prompt = self._create_analyst_prompt(example)
                analysis_output = self._run_agent_step(model, tokenizer, analyst_prompt)

                # --- Passo 2: Agente Pesquisador ---
                search_prompt = self._create_search_prompt(analysis_output, example['context'])
                search_output = self._run_agent_step(model, tokenizer, search_prompt)

                # --- Passo 3: Agente Sintetizador ---
                synthesis_prompt = self._create_synthesis_prompt(search_output, example['question'])
                final_answer = self._run_agent_step(model, tokenizer, synthesis_prompt)

                # Limpeza final da resposta
                if "não sei" in final_answer.lower():
                    final_answer = ""  # SQuAD espera string vazia para "não sei"

            except Exception as e:
                print(f"Erro no workflow do agente: {e}")
                final_answer = ""  # Falha segura

            predictions.append({'id': example['id'], 'prediction_text': final_answer})
            references.append({'id': example['id'], 'answers': example['answers']})

        return metrics_calc.compute(predictions=predictions, references=references)


    def _create_analyst_prompt(self, example):
        """Cria prompt para o agente analista"""
        return f"""Você é um agente analista especializado em análise de perguntas.

    Analise a seguinte pergunta e contexto:

    Contexto: {example['context']}
    Pergunta: {example['question']}

    Forneça uma análise estruturada identificando:
    1. Tipo de pergunta (factual, inferencial, etc.)
    2. Palavras-chave importantes
    3. Estratégia de busca recomendada

    Análise:"""

    def _create_search_prompt(self, analysis_output, context):
        """Cria prompt para o agente pesquisador"""
        return f"""Você é um agente pesquisador especializado em extrair informações.

    Com base na análise anterior: {analysis_output}

    Examine cuidadosamente o seguinte contexto e extraia as informações mais relevantes:

    Contexto: {context}

    Informações extraídas:"""

    def _create_synthesis_prompt(self, search_output, question):
        """Cria prompt para o agente sintetizador"""
        return f"""Você é um agente sintetizador especializado em formar respostas finais.

    Com base nas informações extraídas: {search_output}

    Responda de forma precisa à pergunta: {question}

    Se as informações não forem suficientes para responder, responda "não sei".

    Resposta final:"""

    def _run_agent_step(self, model, tokenizer, prompt):
        """Executa um passo do workflow de agentes"""
        try:
            inputs = tokenizer(
                prompt,
                return_tensors="pt",
                max_length=self.config.MAX_LENGTH,
                truncation=True
            ).to(self.config.DEVICE)

            with torch.no_grad():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=100,
                    temperature=0.3,
                    do_sample=True,
                    pad_token_id=tokenizer.eos_token_id
                )

            response = tokenizer.decode(outputs[0], skip_special_tokens=True)
            return response[len(prompt):].strip()
        except Exception as e:
            print(f"Erro no passo do agente: {e}")
            return ""



### `ScientificEvaluator`  
Sistema integrado para avaliação científica completa de múltiplos modelos, aplicando várias estratégias de prompt (zero-shot, few-shot, CoT, auto-consistência e workflow multiagente), coletando métricas detalhadas, tratando erros e registrando os resultados para análise.  


In [17]:
class ScientificEvaluator:
    """Sistema completo de avaliação científica"""

    def __init__(self, config, logger):
        self.config = config
        self.logger = logger
        self.prompt_engineer = PromptEngineer(config, logger)

    def comprehensive_evaluation(self, models_dict, datasets):
        """
        Avaliação científica completa de todos os modelos

        Args:
            models_dict: Dict com {nome: (modelo, tokenizer)}
            datasets: Dict com datasets preparados
        """
        print("Iniciando Avaliação Científica Completa...")

        results_matrix = {}
        detailed_results = {}

        strategies = {
            'Zero-Shot': (self.prompt_engineer.evaluate_with_prompt_strategy,
                         {'prompt_func': self.prompt_engineer.zero_shot_prompt}),
            '3-Shot ICL': (self.prompt_engineer.evaluate_with_prompt_strategy,
                          {'prompt_func': self.prompt_engineer.few_shot_prompt,
                           'icl_examples': datasets['icl_examples'], 'num_shots': 3}),
            'Zero-Shot CoT': (self.prompt_engineer.evaluate_with_prompt_strategy,
                             {'prompt_func': self.prompt_engineer.cot_prompt}),
            'Self-Consistency': (self.prompt_engineer.self_consistency_evaluation, {}),
            'Workflow Agente-Executor': (self.prompt_engineer.agentic_workflow_evaluation, {})
        }

        for model_name, (model, tokenizer) in models_dict.items():
            print(f"\n🔬 Avaliando modelo: {model_name}")
            model.to(self.config.DEVICE)

            results_matrix[model_name] = {}
            detailed_results[model_name] = {}

            for strategy_name, (eval_func, kwargs) in strategies.items():
                print(f"  📋 Estratégia: {strategy_name}")

                try:
                    if strategy_name in ['Self-Consistency', 'Workflow Agente-Executor']:
                        # Estas funções têm assinaturas diferentes
                        results = eval_func(model, tokenizer, datasets['evaluation'],
                                        datasets['metrics_calculator'], **kwargs)
                    else:
                        results = eval_func(model, tokenizer, datasets['evaluation'],
                                        datasets['metrics_calculator'], strategy_name, **kwargs)

                    # Armazenar resultados
                    results_matrix[model_name][strategy_name] = {
                        'f1': results['f1'],
                        'exact_match': results['exact_match']
                    }
                    detailed_results[model_name][strategy_name] = results

                    print(f"    ✅ F1: {results['f1']:.2f}, EM: {results['exact_match']:.2f}")

                except Exception as e:
                    print(f"    ❌ Erro: {e}")
                    results_matrix[model_name][strategy_name] = {'f1': 0.0, 'exact_match': 0.0}
                    detailed_results[model_name][strategy_name] = {'error': str(e)}

        # Log da avaliação
        self.logger.log_phase("COMPREHENSIVE_EVALUATION", {
            'status': 'Concluída',
            'models_evaluated': list(models_dict.keys()),
            'strategies_tested': list(strategies.keys()),
            'results_summary': results_matrix
        })

        return results_matrix, detailed_results

### `ScientificReporter`  
Gera relatórios científicos completos a partir dos resultados experimentais, incluindo:

- Conversão de métricas em tabelas (`DataFrame`)
- Análise estatística (médias, desvios, melhorias relativas)
- Geração de visualizações (heatmaps, gráficos de barras, boxplots e curvas de treino)
- Produção de relatório textual detalhado em Markdown com interpretações, conclusões e recomendações
- Registro e salvamento automático dos resultados, figuras e métricas para reprodutibilidade científica


In [18]:
class ScientificReporter:
    """Gerador de relatórios científicos"""

    def __init__(self, config, logger):
        self.config = config
        self.logger = logger

    def generate_comprehensive_report(self, results_matrix, detailed_results,
                                    training_metrics=None, dataset_stats=None):
        """
        Gera relatório científico completo com análises estatísticas
        """
        print("📝 Gerando Relatório Científico...")

        # Converter para DataFrame
        df_results = self._results_to_dataframe(results_matrix, 'f1')
        df_em = self._results_to_dataframe(results_matrix, 'exact_match')

        # Análises estatísticas
        statistical_analysis = self._perform_statistical_analysis(df_results, df_em)

        # Gerar visualizações
        self._generate_visualizations(df_results, df_em, training_metrics)

        # Relatório em texto
        report = self._generate_text_report(df_results, df_em, statistical_analysis,
                                          dataset_stats, training_metrics)

        # Salvar relatório
        with open(f"{self.config.REPORTS_DIR}/scientific_report.md", 'w', encoding='utf-8') as f:
            f.write(report)

        # Salvar dados brutos
        df_results.to_csv(f"{self.config.REPORTS_DIR}/f1_results.csv")
        df_em.to_csv(f"{self.config.REPORTS_DIR}/exact_match_results.csv")

        print("✅ Relatório científico salvo em:", self.config.REPORTS_DIR)

        return report

    def _results_to_dataframe(self, results_matrix, metric):
        """Converte matriz de resultados para DataFrame"""
        data = {}
        for model, strategies in results_matrix.items():
            data[model] = [strategies[strat][metric] for strat in strategies.keys()]

        return pd.DataFrame(data, index=list(next(iter(results_matrix.values())).keys()))

    def _perform_statistical_analysis(self, df_f1, df_em):
        """Realiza análises estatísticas dos resultados"""
        analysis = {}

        # Melhores performers
        analysis['best_model_f1'] = df_f1.mean(axis=0).idxmax()
        analysis['best_strategy_f1'] = df_f1.mean(axis=1).idxmax()
        analysis['best_combination_f1'] = df_f1.stack().idxmax()
        analysis['best_f1_score'] = df_f1.max().max()

        # Variabilidade
        analysis['model_variability'] = df_f1.std(axis=0).to_dict()
        analysis['strategy_variability'] = df_f1.std(axis=1).to_dict()

        # Comparações
        baseline_scores = df_f1.iloc[0]  # Primeira estratégia como baseline
        improvements = {}
        for idx, strategy in enumerate(df_f1.index[1:], 1):
            improvements[strategy] = (df_f1.iloc[idx] - baseline_scores).mean()
        analysis['avg_improvements'] = improvements

        return analysis

    def _generate_visualizations(self, df_f1, df_em, training_metrics=None):
        """Gera visualizações científicas"""
        plt.style.use('seaborn-v0_8')

        # 1. Heatmap de resultados F1
        plt.figure(figsize=(12, 8))
        sns.heatmap(df_f1, annot=True, cmap='YlOrRd', fmt='.2f',
                    cbar_kws={'label': 'F1-Score'})
        plt.title('Matriz de Performance (F1-Score)\nModelos vs Estratégias de Prompt')
        plt.tight_layout()
        plt.savefig(f"{self.config.FIGURES_DIR}/f1_heatmap.png", dpi=300)
        plt.close()

        # 2. Gráfico de barras comparativo
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

        # F1-Score por modelo
        df_f1.mean(axis=0).plot(kind='bar', ax=ax1, color='skyblue')
        ax1.set_title('Performance Média por Modelo (F1-Score)')
        ax1.set_ylabel('F1-Score')
        ax1.tick_params(axis='x', rotation=45)

        # F1-Score por estratégia
        df_f1.mean(axis=1).plot(kind='bar', ax=ax2, color='lightcoral')
        ax2.set_title('Performance Média por Estratégia (F1-Score)')
        ax2.set_ylabel('F1-Score')
        ax2.tick_params(axis='x', rotation=45)

        plt.tight_layout()
        plt.savefig(f"{self.config.FIGURES_DIR}/comparative_bars.png", dpi=300)
        plt.close()

        # 3. Box plot de distribuições
        plt.figure(figsize=(12, 6))
        df_f1.T.boxplot()
        plt.title('Distribuição de F1-Scores por Estratégia')
        plt.ylabel('F1-Score')
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.savefig(f"{self.config.FIGURES_DIR}/f1_distributions.png", dpi=300)
        plt.close()

        # 4. Curvas de treinamento (se disponível)
        if training_metrics:
            self._plot_training_curves(training_metrics)

    def _plot_training_curves(self, training_metrics):
        """Plota curvas de treinamento para análise de convergência"""
        fig, axes = plt.subplots(2, 2, figsize=(15, 10))

        for idx, (technique, metrics) in enumerate(training_metrics.items()):
            if idx >= 4:  # Máximo 4 técnicas
                break

            ax = axes[idx//2, idx%2]
            epochs = metrics.get('epochs', [])

            if 'kd_losses' in metrics:
                ax.plot(epochs, metrics['kd_losses'], 'b-', label='KD Loss', marker='o')
            if 'ce_losses' in metrics:
                ax.plot(epochs, metrics['ce_losses'], 'r--', label='CE Loss', marker='s')
            if 'total_losses' in metrics:
                ax.plot(epochs, metrics['total_losses'], 'g-', label='Total Loss', marker='^')

            ax.set_title(f'Curvas de Treinamento - {technique}')
            ax.set_xlabel('Época')
            ax.set_ylabel('Loss')
            ax.legend()
            ax.grid(True, alpha=0.3)

        plt.tight_layout()
        plt.savefig(f"{self.config.FIGURES_DIR}/training_curves.png", dpi=300)
        plt.close()

    def _generate_text_report(self, df_f1, df_em, statistical_analysis,
                            dataset_stats, training_metrics):
        """Gera relatório científico em texto"""

        report = f"""# Relatório Científico: Aprimoramento de Small Language Models

**Data:** {datetime.now().strftime('%d/%m/%Y %H:%M')}
**Modelo Base:** {self.config.STUDENT_MODEL_ID}
**Ambiente:** {'GPU' if self.config.DEVICE == 'cuda' else 'CPU'}

## 1. Resumo Executivo

Este estudo investiga técnicas de Knowledge Distillation (KD) combinadas com estratégias avançadas de Prompt Engineering em Small Language Models (SLMs) para a tarefa de Question Answering, utilizando o dataset SQuAD em ambiente computacional limitado.

### Principais Resultados:
- **Melhor Modelo:** {statistical_analysis['best_model_f1']} (F1: {df_f1[statistical_analysis['best_model_f1']].mean():.2f})
- **Melhor Estratégia:** {statistical_analysis['best_strategy_f1']} (F1: {df_f1.mean(axis=1)[statistical_analysis['best_strategy_f1']]:.2f})
- **Melhor Combinação:** {statistical_analysis['best_combination_f1'][1]} + {statistical_analysis['best_combination_f1'][0]} (F1: {statistical_analysis['best_f1_score']:.2f})

## 2. Metodologia

### 2.1 Dataset e Configuração Experimental"""

        if dataset_stats:
            report += f"""
- **Dataset:** SQuAD (Stanford Question Answering Dataset)
- **Tamanho do conjunto de avaliação:** {self.config.EVAL_SUBSET_SIZE} exemplos
- **Tamanho do conjunto de KD:** {self.config.KD_TRANSFER_SIZE} exemplos
- **Comprimento médio do contexto:** {dataset_stats['avg_context_length']:.1f} ± {dataset_stats['std_context_length']:.1f} palavras
- **Comprimento médio da pergunta:** {dataset_stats['avg_question_length']:.1f} ± {dataset_stats['std_question_length']:.1f} palavras"""

        report += f"""

### 2.2 Técnicas de Knowledge Distillation Testadas:
1. **KD Base:** Destilação tradicional baseada em logits
2. **Auto-Destilação:** KD usando modelo especialista interno
3. **Destilação de Explicações:** KD baseada em raciocínio textual

### 2.3 Estratégias de Prompt Engineering:
1. **Zero-Shot:** Prompt direto sem exemplos
2. **3-Shot ICL:** In-Context Learning com 3 exemplos
3. **Zero-Shot CoT:** Chain-of-Thought sem exemplos
4. **Self-Consistency:** Múltiplos caminhos com votação majoritária

### 2.4 Hiperparâmetros:
- **Temperatura KD:** {self.config.TEMPERATURE}
- **Taxa de Aprendizado:** {self.config.LEARNING_RATE}
- **Épocas:** {self.config.NUM_EPOCHS}
- **LoRA Rank:** {self.config.LORA_R}

## 3. Resultados Experimentais

### 3.1 Matriz de Performance (F1-Score)
"""

        report += "\n" + df_f1.to_markdown(floatfmt='.2f') + "\n"

        report += f"""
### 3.2 Matriz de Performance (Exact Match)
"""

        report += "\n" + df_em.to_markdown(floatfmt='.2f') + "\n"

        report += f"""
### 3.3 Análise Estatística

#### 3.3.1 Performance por Modelo:
"""

        for model in df_f1.columns:
            mean_f1 = df_f1[model].mean()
            std_f1 = df_f1[model].std()
            report += f"- **{model}:** {mean_f1:.2f} ± {std_f1:.2f}\n"

        report += f"""
#### 3.3.2 Performance por Estratégia:
"""

        for strategy in df_f1.index:
            mean_f1 = df_f1.loc[strategy].mean()
            std_f1 = df_f1.loc[strategy].std()
            report += f"- **{strategy}:** {mean_f1:.2f} ± {std_f1:.2f}\n"

        report += f"""
#### 3.3.3 Melhorias Relativas ao Baseline:
"""

        for strategy, improvement in statistical_analysis['avg_improvements'].items():
            report += f"- **{strategy}:** {improvement:+.2f} pontos F1\n"

        if training_metrics:
            report += f"""
### 3.4 Análise de Convergência do Treinamento

Os modelos KD mostraram convergência adequada durante o treinamento:
"""
            for technique, metrics in training_metrics.items():
                if 'final_kd_loss' in metrics:
                    report += f"- **{technique}:** Loss KD final = {metrics['final_kd_loss']:.4f}\n"

        report += f"""
## 4. Discussão e Interpretação

### 4.1 Eficácia das Técnicas de Knowledge Distillation
"""

        # Análise das técnicas KD
        kd_models = [col for col in df_f1.columns if 'KD' in col or 'Auto' in col or 'Explicações' in col]
        if kd_models:
            best_kd = df_f1[kd_models].mean().idxmax()
            best_kd_score = df_f1[kd_models].mean().max()
            baseline_score = df_f1.iloc[:, 0].mean() if len(df_f1.columns) > 0 else 0

            report += f"""
A técnica **{best_kd}** apresentou melhor performance média (F1: {best_kd_score:.2f}), representando uma melhoria de {best_kd_score - baseline_score:+.2f} pontos sobre o modelo base.

### 4.2 Impacto das Estratégias de Prompt Engineering

A estratégia **{statistical_analysis['best_strategy_f1']}** demonstrou maior eficácia, sugerindo que {self._interpret_strategy(statistical_analysis['best_strategy_f1'])}.
"""

        report += f"""
### 4.3 Sinergia entre KD e Prompt Engineering

A combinação **{statistical_analysis['best_combination_f1'][1]} + {statistical_analysis['best_combination_f1'][0]}** alcançou o melhor resultado (F1: {statistical_analysis['best_f1_score']:.2f}), demonstrando sinergia positiva entre as técnicas.

### 4.4 Limitações do Estudo

1. **Tamanho do dataset:** Subset limitado para avaliação rápida
2. **Recursos computacionais:** Configuração LoRA para economia de memória
3. **Modelos:** Foco em SLMs para ambiente restrito
4. **Domínio:** Limitado à tarefa de QA com SQuAD

## 5. Conclusões e Trabalhos Futuros

### 5.1 Conclusões Principais:

1. **Knowledge Distillation é eficaz** para SLMs em ambiente limitado
2. **Prompt Engineering complementa KD**, especialmente técnicas CoT
3. **Combinações específicas** mostram sinergia significativa
4. **Auto-destilação** pode ser viável quando recursos são limitados

### 5.2 Recomendações:

1. Avaliar em datasets maiores e múltiplos domínios
2. Testar modelos teachers mais diversos
3. Investigar técnicas híbridas KD + fine-tuning
4. Implementar avaliação qualitativa das explicações

### 5.3 Contribuições Científicas:

- Metodologia sistemática para avaliação combinada KD + Prompt Engineering
- Análise quantitativa de sinergia entre técnicas
- Framework reproduzível para ambiente computacional limitado

## 6. Reprodutibilidade

Todo o código, configurações e resultados estão disponíveis em:
- **Código:** `{self.config.OUTPUT_DIR}/`
- **Modelos:** `{self.config.MODELS_DIR}/`
- **Dados:** `{self.config.REPORTS_DIR}/`
- **Figuras:** `{self.config.FIGURES_DIR}/`

**Seed:** 42 (para reprodutibilidade)
"""

        return report

    def _interpret_strategy(self, strategy_name):
        """Interpreta o resultado da melhor estratégia"""
        interpretations = {
            'Zero-Shot': 'instruções diretas são suficientes para SLMs treinados',
            '3-Shot ICL': 'aprendizado contextual melhora significativamente a performance',
            'Zero-Shot CoT': 'raciocínio explícito é benéfico mesmo sem exemplos',
            'Self-Consistency': 'múltiplos caminhos de raciocínio aumentam a confiabilidade'
        }
        return interpretations.get(strategy_name, 'essa abordagem é particularmente eficaz')

### `ScientificExperimentRunner`  
Orquestrador central de experimentos científicos. Gerencia todas as fases:  
1. **Análise de dados** com `DatasetManager`  
2. **Treinamento com distilação de conhecimento** via `KnowledgeDistiller` (KD tradicional, auto-distilação, etc.)  
3. **Avaliação científica** com múltiplas estratégias de prompting usando `ScientificEvaluator`  
4. **Geração de relatório** completo e visualizações com `ScientificReporter`  

Inclui mecanismos robustos de tratamento de erro, carregamento seguro de modelos (`_load_model_safe`) e reutilização de modelos salvos para evitar retrabalho.  
Salva logs detalhados de todas as fases para rastreabilidade científica.


In [19]:
class ScientificExperimentRunner:
    """Orquestrador principal do experimento científico"""

    def __init__(self):
        self.config = ExperimentConfig()
        self.logger = ScientificLogger(self.config)
        self.dataset_manager = DatasetManager(self.config, self.logger)
        self.distiller = KnowledgeDistiller(self.config, self.logger)
        self.evaluator = ScientificEvaluator(self.config, self.logger)
        self.reporter = ScientificReporter(self.config, self.logger)

    def run_complete_experiment(self, run_training=False, run_evaluation=True):
        """
        Executa o experimento científico completo

        Args:
            run_training: Se True, executa fase de treinamento KD
            run_evaluation: Se True, executa avaliação completa
        """
        print("INICIANDO EXPERIMENTO CIENTÍFICO COMPLETO")
        print("=" * 60)

        try:
            # Fase 0: Setup e análise de dados
            datasets, dataset_stats = self.dataset_manager.load_and_analyze_data()

            # Dicionário para armazenar modelos treinados
            trained_models = {}
            training_metrics = {}

            # Fase 1: Treinamento KD (opcional)
            if run_training:
                print("\n🔬 FASE 1: KNOWLEDGE DISTILLATION")
                print("-" * 40)

                # Carregar modelos base
                student_model, student_tokenizer = self._load_model_safe(self.config.STUDENT_MODEL_ID)
                teacher_model, teacher_tokenizer = self._load_model_safe(self.config.TEACHER_MODEL_ID)

                if student_model and teacher_model:
                    # 1. KD Base
                    kd_base_model, kd_base_metrics = self.distiller.distill_with_logits(
                        student_model, teacher_model, student_tokenizer, teacher_tokenizer,
                        datasets['kd_transfer'], "KD_Base"
                    )
                    trained_models['KD Base'] = (kd_base_model, student_tokenizer)
                    training_metrics['KD Base'] = kd_base_metrics

                    # 2. Auto-Destilação
                    student_model2, _ = self._load_model_safe(self.config.STUDENT_MODEL_ID)
                    auto_dist_model, auto_dist_metrics = self.distiller.self_distillation(
                        student_model2, student_tokenizer, datasets['kd_transfer']
                    )
                    trained_models['KD Auto-Destilado'] = (auto_dist_model, student_tokenizer)
                    training_metrics['KD Auto-Destilado'] = auto_dist_metrics

                    # 3. Destilação de Explicações (simplificada)
                    # Nota: Implementação completa requer geração de dataset explicativo
                    print("⚠️ Destilação de Explicações: usando fine-tuning padrão como aproximação")
                else:
                    print("❌ Erro ao carregar modelos - pulando fase de treinamento")

            # Carregar modelos para avaliação
            models_for_evaluation = self._prepare_evaluation_models(trained_models)

            # Fase 2: Avaliação
            if run_evaluation and models_for_evaluation:
                print("\nFASE 2: AVALIAÇÃO CIENTÍFICA")
                print("-" * 40)

                results_matrix, detailed_results = self.evaluator.comprehensive_evaluation(
                    models_for_evaluation, datasets
                )

                # Fase 3: Relatório
                print("\n📝 FASE 3: GERAÇÃO DE RELATÓRIO")
                print("-" * 40)

                report = self.reporter.generate_comprehensive_report(
                    results_matrix, detailed_results, training_metrics, dataset_stats
                )

                print("\n✅ EXPERIMENTO CONCLUÍDO COM SUCESSO!")
                print(f"📁 Resultados salvos em: {self.config.OUTPUT_DIR}")

                return {
                    'results_matrix': results_matrix,
                    'detailed_results': detailed_results,
                    'training_metrics': training_metrics,
                    'report': report
                }

        except Exception as e:
            error_msg = f"❌ Erro durante execução: {e}"
            print(error_msg)
            self.logger.log_phase("ERROR", {'error': str(e)})
            raise

        finally:
            # Salvar log do experimento
            self.logger.save_experiment_log()

    def _load_model_safe(self, model_id, retries=2):
        """Carrega modelo com múltiplas tentativas e fallbacks"""
        for attempt in range(retries):
            try:
                print(f"🔄 Carregando {model_id} (tentativa {attempt+1}/{retries})...")

                # Configurações mais conservadoras
                tokenizer = AutoTokenizer.from_pretrained(
                    model_id,
                    trust_remote_code=True,
                    use_fast=True
                )

                if tokenizer.pad_token is None:
                    tokenizer.pad_token = tokenizer.eos_token

                # Configuração de loading mais robusta
                model_kwargs = {
                    "trust_remote_code": True,
                    "low_cpu_mem_usage": True,
                }

                # Ajustar configurações baseado na disponibilidade de GPU
                if torch.cuda.is_available():
                    model_kwargs.update({
                        "device_map": "auto",
                        "torch_dtype": torch.bfloat16,
                    })
                else:
                    model_kwargs["torch_dtype"] = torch.float32

                model = AutoModelForCausalLM.from_pretrained(model_id, **model_kwargs)

                print(f"✅ {model_id} carregado com sucesso")
                return model, tokenizer

            except Exception as e:
                print(f"❌ Tentativa {attempt+1} falhou para {model_id}: {e}")
                if attempt == retries - 1:
                    print(f"❌ Falha final ao carregar {model_id}")
                    return None, None

                # Limpar cache entre tentativas
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

        return None, None

    def _prepare_evaluation_models(self, trained_models):
        """Prepara modelos para avaliação"""
        models_dict = {}

        # Modelo base
        base_model, base_tokenizer = self._load_model_safe(self.config.STUDENT_MODEL_ID)
        if base_model:
            models_dict['SLM Base'] = (base_model, base_tokenizer)

        # Modelos treinados
        for name, (model, tokenizer) in trained_models.items():
            models_dict[name] = (model, tokenizer)

        # Tentar carregar modelos salvos se treinamento não foi executado
        if not trained_models:
            for model_name in ['KD Base', 'KD Auto-Destilado', 'KD com Explicações']:
                model_path = f"{self.config.MODELS_DIR}/{model_name.lower().replace(' ', '_')}"
                if os.path.exists(model_path):
                    try:
                        base_for_peft, tokenizer = self._load_model_safe(self.config.STUDENT_MODEL_ID)
                        if base_for_peft:
                            peft_model = PeftModel.from_pretrained(base_for_peft, model_path)
                            merged_model = peft_model.merge_and_unload()
                            models_dict[model_name] = (merged_model, tokenizer)
                            print(f"✅ Modelo {model_name} carregado de {model_path}")
                    except Exception as e:
                        print(f"❌ Erro ao carregar {model_name}: {e}")

        return models_dict

## Função principal de execução
Função para executar o experimento científico, com opções para treinamento e avaliação.

In [20]:
# Função principal para execução
def run_scientific_experiment(training=False, evaluation=True, debug_mode=True):
    """
    Função principal para executar o experimento científico com melhor handling de erros
    """
    try:
        runner = ScientificExperimentRunner()

        if debug_mode:
            # Configurações mais conservadoras para debug
            runner.config.EVAL_SUBSET_SIZE = min(50, runner.config.EVAL_SUBSET_SIZE)
            runner.config.KD_TRANSFER_SIZE = min(200, runner.config.KD_TRANSFER_SIZE)
            runner.config.NUM_EPOCHS = 1
            runner.config.BATCH_SIZE = 1
            print("🐛 Modo DEBUG ativado - configurações reduzidas")

        print("CONFIGURAÇÃO DO EXPERIMENTO")
        print(f"Modelo Estudante: {runner.config.STUDENT_MODEL_ID}")
        print(f"Modelo Professor: {runner.config.TEACHER_MODEL_ID}")
        print(f"Dispositivo: {runner.config.DEVICE}")
        print(f"Tamanho Avaliação: {runner.config.EVAL_SUBSET_SIZE}")
        print(f"Treinamento: {'✅' if training else '❌'}")
        print(f"Avaliação: {'✅' if evaluation else '❌'}")
        print("=" * 60)

        return runner.run_complete_experiment(
            run_training=training,
            run_evaluation=evaluation
        )

    except Exception as e:
        print(f"❌ Erro crítico na execução: {e}")
        print("💡 Tente executar com debug_mode=True para configurações mais conservadoras")
        return None

In [ ]:
# AgenticWorkflow Integration with SLM_Final Notebook
# This file contains the agentic AI extensions that integrate with existing classes

import torch
import numpy as np
from tqdm import tqdm
import re

class AgenticAIFramework:
    """
    Framework Agentic AI avançado para SLMs - Versão Integrada
    Baseado em: https://arxiv.org/abs/2505.10468
    """
    
    def __init__(self, config, logger):
        self.config = config
        self.logger = logger
        self.agent_registry = {}
        self.conversation_memory = []
        self.tool_registry = {}
        
    def register_agent(self, agent_name, agent_config):
        """Registra um agente no framework"""
        self.agent_registry[agent_name] = {
            'role': agent_config.get('role', 'general'),
            'specialization': agent_config.get('specialization', []),
            'prompt_template': agent_config.get('prompt_template', ''),
            'max_iterations': agent_config.get('max_iterations', 3),
            'confidence_threshold': agent_config.get('confidence_threshold', 0.7)
        }
        
    def register_tool(self, tool_name, tool_func, tool_description):
        """Registra uma ferramenta que os agentes podem usar"""
        self.tool_registry[tool_name] = {
            'function': tool_func,
            'description': tool_description
        }

class AdvancedAgenticPromptEngineering:
    """
    Extensão das capacidades de engenharia de prompt com Agentic AI
    Projetado para integrar com as classes existentes do SLM_Final
    """
    
    def __init__(self, config, logger):
        self.config = config
        self.logger = logger
        self.agentic_framework = AgenticAIFramework(config, logger)
        self._setup_agents()
        self._setup_tools()
    
    def _setup_agents(self):
        """Configura agentes especializados baseados no paper Agentic AI"""
        
        # Agente Planejador
        self.agentic_framework.register_agent('planner', {
            'role': 'task_decomposition',
            'specialization': ['planning', 'reasoning', 'task_breakdown'],
            'prompt_template': '''Você é um agente planejador especializado em decomposição de tarefas.

Contexto: {context}
Pergunta: {question}

Crie um plano estruturado para responder esta pergunta:
1. Identifique as informações-chave necessárias
2. Determine a estratégia de busca no contexto
3. Identifique potenciais armadilhas ou ambiguidades
4. Planeje a estrutura da resposta
5. Defina critérios de validação

Plano:''',
            'max_iterations': 1,
            'confidence_threshold': 0.8
        })
        
        # Agente Pesquisador
        self.agentic_framework.register_agent('researcher', {
            'role': 'information_extraction',
            'specialization': ['search', 'extraction', 'fact_finding'],
            'prompt_template': '''Você é um agente pesquisador especializado em extração de informações.

Plano recebido: {plan}
Contexto para pesquisa: {context}
Pergunta específica: {question}

Extraia TODAS as informações relevantes do contexto:
- Fatos diretos relacionados à pergunta
- Informações que podem ser inferidas
- Relações entre diferentes partes do texto
- Possíveis respostas alternativas

FATOS DIRETOS: [lista]
INFERÊNCIAS: [lista]
RELAÇÕES: [lista]
CANDIDATOS À RESPOSTA: [lista]''',
            'max_iterations': 2,
            'confidence_threshold': 0.75
        })
        
        # Agente Sintetizador
        self.agentic_framework.register_agent('synthesizer', {
            'role': 'synthesis_reasoning',
            'specialization': ['synthesis', 'reasoning', 'answer_generation'],
            'prompt_template': '''Você é um agente sintetizador especializado em raciocínio e síntese.

Plano original: {plan}
Informações extraídas: {research_results}
Pergunta: {question}

Sintetize as informações para gerar uma resposta precisa:
1. Avalie a relevância de cada informação extraída
2. Identifique a resposta mais provável
3. Considere informações conflitantes
4. Verifique consistência com o contexto

RACIOCÍNIO: [explique o processo]
RESPOSTA FINAL: [resposta concisa]''',
            'max_iterations': 1,
            'confidence_threshold': 0.8
        })
        
        # Agente Verificador
        self.agentic_framework.register_agent('verifier', {
            'role': 'quality_control',
            'specialization': ['verification', 'validation', 'quality_check'],
            'prompt_template': '''Você é um agente verificador especializado em controle de qualidade.

Pergunta original: {question}
Contexto: {context}
Resposta proposta: {proposed_answer}

Verifique a qualidade da resposta:
1. PRECISÃO: A resposta está correta com base no contexto?
2. COMPLETUDE: A resposta aborda totalmente a pergunta?
3. CONSISTÊNCIA: A resposta é internamente consistente?
4. RELEVÂNCIA: A resposta é diretamente relevante à pergunta?

PRECISÃO: [1-5] - [justificativa]
COMPLETUDE: [1-5] - [justificativa]  
CONSISTÊNCIA: [1-5] - [justificativa]
RELEVÂNCIA: [1-5] - [justificativa]

NOTA GERAL: [média]
RECOMENDAÇÃO: [aceitar/revisar/rejeitar]''',
            'max_iterations': 1,
            'confidence_threshold': 0.7
        })
    
    def _setup_tools(self):
        """Configura ferramentas que os agentes podem usar"""
        
        # Ferramenta de análise de similaridade semântica (com fallback)
        def semantic_similarity_tool(text1, text2):
            """Calcula similaridade semântica entre dois textos"""
            try:
                from sklearn.feature_extraction.text import TfidfVectorizer
                from sklearn.metrics.pairwise import cosine_similarity
                
                vectorizer = TfidfVectorizer()
                vectors = vectorizer.fit_transform([text1, text2])
                similarity = cosine_similarity(vectors[0:1], vectors[1:2])[0][0]
                return f"Similaridade semântica: {similarity:.3f}"
            except ImportError:
                # Fallback: simple word overlap similarity
                words1 = set(text1.lower().split())
                words2 = set(text2.lower().split())
                intersection = len(words1 & words2)
                union = len(words1 | words2)
                similarity = intersection / union if union > 0 else 0.0
                return f"Similaridade semântica (fallback): {similarity:.3f}"
            except Exception:
                return "Ferramenta de similaridade não disponível"
        
        self.agentic_framework.register_tool(
            'semantic_similarity',
            semantic_similarity_tool,
            'Calcula similaridade semântica entre dois textos'
        )
        
        # Ferramenta de extração de entidades nomeadas
        def named_entity_extraction_tool(text):
            """Extrai possíveis entidades nomeadas do texto"""
            patterns = {
                'DATAS': re.findall(r'\b\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\b|\b\d{4}\b', text),
                'NÚMEROS': re.findall(r'\b\d+(?:\.\d+)?\b', text),
                'NOMES_PRÓPRIOS': re.findall(r'\b[A-Z][a-z]+(?:\s+[A-Z][a-z]+)*\b', text),
                'LOCAIS': re.findall(r'\b(?:em|de|para|até)\s+([A-Z][a-z]+(?:\s+[A-Z][a-z]+)*)\b', text)
            }
            
            result = "ENTIDADES ENCONTRADAS:\n"
            for categoria, entidades in patterns.items():
                if entidades:
                    result += f"{categoria}: {list(set(entidades))}\n"
            
            return result
        
        self.agentic_framework.register_tool(
            'named_entity_extraction',
            named_entity_extraction_tool,
            'Extrai entidades nomeadas do texto (datas, números, nomes, locais)'
        )

# EXTENSÕES PARA AS CLASSES EXISTENTES DO SLM_FINAL

def add_agentic_methods_to_prompt_engineer():
    """
    Adiciona métodos agenticos à classe PromptEngineer existente
    Use esta função após importar as classes do SLM_Final
    """
    
    def agentic_multiagent_evaluation(self, model, tokenizer, eval_dataset, metrics_calc):
        """
        Avaliação usando framework multiagente avançado
        Integrado com a classe PromptEngineer existente
        """
        print("🤖 Avaliação Multiagente Agentic")
        
        # Inicializar framework agentic se não existir
        if not hasattr(self, 'agentic_framework'):
            self.agentic_prompt_engineer = AdvancedAgenticPromptEngineering(self.config, self.logger)
        
        model.eval()
        predictions = []
        references = []
        detailed_logs = []
        
        # Limitar para subset para eficiência
        eval_subset = eval_dataset.select(range(min(20, len(eval_dataset))))
        
        for idx, example in enumerate(tqdm(eval_subset, desc="Agentic Multi-Agent")):
            try:
                example_log = {
                    'example_id': example['id'],
                    'question': example['question'],
                    'agents_outputs': {},
                    'final_confidence': 0.0
                }
                
                # === FASE 1: PLANEJAMENTO ===
                planner_output = self._run_agent_with_tools(
                    model, tokenizer, 'planner', {
                        'context': example['context'][:1000],
                        'question': example['question']
                    }, example_log
                )
                
                # === FASE 2: PESQUISA ===
                research_output = self._run_agent_with_tools(
                    model, tokenizer, 'researcher', {
                        'plan': planner_output,
                        'context': example['context'],
                        'question': example['question']
                    }, example_log
                )
                
                # === FASE 3: SÍNTESE ===
                synthesis_output = self._run_agent_with_tools(
                    model, tokenizer, 'synthesizer', {
                        'plan': planner_output,
                        'research_results': research_output,
                        'question': example['question']
                    }, example_log
                )
                
                final_answer = self._extract_final_answer_from_synthesis(synthesis_output)
                
                # === FASE 4: VERIFICAÇÃO ===
                verification_output = self._run_agent_with_tools(
                    model, tokenizer, 'verifier', {
                        'question': example['question'],
                        'context': example['context'][:500],
                        'proposed_answer': final_answer
                    }, example_log
                )
                
                # Avaliar aceitação
                if 'aceitar' in verification_output.lower():
                    verified_answer = final_answer
                    example_log['verification_status'] = 'aceita'
                elif 'revisar' in verification_output.lower():
                    verified_answer = final_answer
                    example_log['verification_status'] = 'revisada'
                else:
                    verified_answer = ""
                    example_log['verification_status'] = 'rejeitada'
                
                confidence = self._extract_confidence_from_verification(verification_output)
                example_log['final_confidence'] = confidence
                
                predictions.append({
                    'id': example['id'],
                    'prediction_text': verified_answer
                })
                
                references.append({
                    'id': example['id'],
                    'answers': example['answers']
                })
                
                detailed_logs.append(example_log)
                
            except Exception as e:
                print(f"Erro no exemplo {idx}: {e}")
                predictions.append({'id': example['id'], 'prediction_text': ""})
                references.append({'id': example['id'], 'answers': example['answers']})
        
        # Calcular métricas
        results = metrics_calc.compute(predictions=predictions, references=references)
        
        # Adicionar métricas agenticas
        valid_logs = [log for log in detailed_logs if 'verification_status' in log]
        if valid_logs:
            results.update({
                'avg_confidence': np.mean([log['final_confidence'] for log in valid_logs]),
                'acceptance_rate': len([log for log in valid_logs if log.get('verification_status') == 'aceita']) / len(valid_logs),
                'rejection_rate': len([log for log in valid_logs if log.get('verification_status') == 'rejeitada']) / len(valid_logs)
            })
        
        print(f"✅ F1: {results['f1']:.2f}, EM: {results['exact_match']:.2f}")
        print(f"🎯 Confiança média: {results.get('avg_confidence', 0):.2f}")
        print(f"✓ Taxa de aceitação: {results.get('acceptance_rate', 0)*100:.1f}%")
        
        return results
    
    def _run_agent_with_tools(self, model, tokenizer, agent_name, inputs, example_log):
        """Executa um agente com acesso a ferramentas"""
        if not hasattr(self, 'agentic_prompt_engineer'):
            self.agentic_prompt_engineer = AdvancedAgenticPromptEngineering(self.config, self.logger)
        
        agent_config = self.agentic_prompt_engineer.agentic_framework.agent_registry[agent_name]
        prompt_template = agent_config['prompt_template']
        
        # Construir prompt
        prompt = prompt_template.format(**inputs)
        
        try:
            inputs_tensor = tokenizer(
                prompt,
                return_tensors="pt",
                max_length=self.config.MAX_LENGTH * 2,
                truncation=True
            ).to(self.config.DEVICE)
            
            with torch.no_grad():
                outputs = model.generate(
                    **inputs_tensor,
                    max_new_tokens=150,
                    temperature=0.3,
                    do_sample=True,
                    pad_token_id=tokenizer.eos_token_id
                )
            
            response = tokenizer.decode(outputs[0], skip_special_tokens=True)
            agent_output = response[len(prompt):].strip()
            
            example_log['agents_outputs'][agent_name] = agent_output
            return agent_output
            
        except Exception as e:
            error_msg = f"Erro no agente {agent_name}: {e}"
            example_log['agents_outputs'][agent_name] = error_msg
            return error_msg
    
    def _extract_final_answer_from_synthesis(self, synthesis_output):
        """Extrai resposta final do output do sintetizador"""
        patterns = [
            r'RESPOSTA FINAL:\s*(.+?)(?:\n|$)',
            r'resposta final[:\s]*(.+?)(?:\n|$)',
            r'RESPOSTA:\s*(.+?)(?:\n|$)'
        ]
        
        for pattern in patterns:
            match = re.search(pattern, synthesis_output, re.IGNORECASE)
            if match:
                return match.group(1).strip()
        
        # Fallback: última linha não vazia
        lines = [line.strip() for line in synthesis_output.split('\n') if line.strip()]
        return lines[-1] if lines else ""
    
    def _extract_confidence_from_verification(self, verification_output):
        """Extrai nível de confiança do verificador"""
        nota_pattern = r'NOTA GERAL:\s*(\d+(?:\.\d+)?)'
        match = re.search(nota_pattern, verification_output, re.IGNORECASE)
        
        if match:
            nota = float(match.group(1))
            return nota / 5.0  # Normalizar para 0-1
        
        # Fallback baseado em palavras-chave
        if 'aceitar' in verification_output.lower():
            return 0.8
        elif 'revisar' in verification_output.lower():
            return 0.6
        elif 'rejeitar' in verification_output.lower():
            return 0.3
        
        return 0.5
    
    def self_reflection_evaluation(self, model, tokenizer, eval_dataset, metrics_calc):
        """
        Implementa auto-reflexão baseada no framework Agentic AI
        """
        print("🧠 Avaliação com Auto-Reflexão")
        
        model.eval()
        predictions = []
        references = []
        reflection_scores = []
        
        # Subset menor para eficiência
        eval_subset = eval_dataset.select(range(min(15, len(eval_dataset))))
        
        for example in tqdm(eval_subset, desc="Self-Reflection"):
            try:
                # === FASE 1: RESPOSTA INICIAL ===
                initial_prompt = f"""Contexto: {example['context']}
Pergunta: {example['question']}

Forneça uma resposta inicial:"""

                inputs = tokenizer(initial_prompt, return_tensors="pt", 
                                 max_length=self.config.MAX_LENGTH, truncation=True).to(self.config.DEVICE)
                
                with torch.no_grad():
                    outputs = model.generate(**inputs, max_new_tokens=60, temperature=0.7, 
                                           do_sample=True, pad_token_id=tokenizer.eos_token_id)
                
                initial_response = tokenizer.decode(outputs[0], skip_special_tokens=True)[len(initial_prompt):].strip()
                
                # === FASE 2: AUTO-REFLEXÃO ===
                reflection_prompt = f"""Contexto: {example['context']}
Pergunta: {example['question']}
Minha resposta inicial: {initial_response}

Reflexão crítica sobre minha resposta:
1. ANÁLISE: Minha resposta está correta e completa?
2. VERIFICAÇÃO: Baseei-me adequadamente no contexto?
3. MELHORIAS: Como posso melhorar?

Reflexão:"""

                reflection_inputs = tokenizer(reflection_prompt, return_tensors="pt",
                                           max_length=self.config.MAX_LENGTH*2, truncation=True).to(self.config.DEVICE)
                
                with torch.no_grad():
                    reflection_outputs = model.generate(**reflection_inputs, max_new_tokens=100,
                                                      temperature=0.5, do_sample=True, 
                                                      pad_token_id=tokenizer.eos_token_id)
                
                reflection = tokenizer.decode(reflection_outputs[0], skip_special_tokens=True)[len(reflection_prompt):].strip()
                
                # === FASE 3: RESPOSTA REFINADA ===
                refinement_prompt = f"""Contexto: {example['context']}
Pergunta: {example['question']}
Resposta inicial: {initial_response}
Minha reflexão: {reflection}

Com base na reflexão, forneço uma resposta final melhorada:"""

                refinement_inputs = tokenizer(refinement_prompt, return_tensors="pt",
                                            max_length=self.config.MAX_LENGTH*2, truncation=True).to(self.config.DEVICE)
                
                with torch.no_grad():
                    final_outputs = model.generate(**refinement_inputs, max_new_tokens=80,
                                                  temperature=0.3, do_sample=True,
                                                  pad_token_id=tokenizer.eos_token_id)
                
                final_response = tokenizer.decode(final_outputs[0], skip_special_tokens=True)[len(refinement_prompt):].strip()
                
                # Avaliar qualidade da reflexão
                reflection_score = self._evaluate_reflection_quality(reflection)
                reflection_scores.append(reflection_score)
                
                predictions.append({'id': example['id'], 'prediction_text': final_response})
                references.append({'id': example['id'], 'answers': example['answers']})
                
            except Exception as e:
                print(f"Erro na auto-reflexão: {e}")
                predictions.append({'id': example['id'], 'prediction_text': ""})
                references.append({'id': example['id'], 'answers': example['answers']})
                reflection_scores.append(0.0)
        
        # Calcular métricas
        results = metrics_calc.compute(predictions=predictions, references=references)
        results.update({
            'avg_reflection_quality': np.mean(reflection_scores),
            'std_reflection_quality': np.std(reflection_scores)
        })
        
        print(f"✅ F1: {results['f1']:.2f}, EM: {results['exact_match']:.2f}")
        print(f"🧠 Qualidade da reflexão: {results['avg_reflection_quality']:.2f}")
        
        return results
    
    def _evaluate_reflection_quality(self, reflection_text):
        """Avalia a qualidade da auto-reflexão"""
        score = 0.0
        reflection_lower = reflection_text.lower()
        
        # Presença de análise crítica
        if any(word in reflection_lower for word in ['análise', 'crítica', 'correto', 'incorreto']):
            score += 0.25
        
        # Verificação de fontes
        if any(word in reflection_lower for word in ['contexto', 'baseado', 'informação', 'fonte']):
            score += 0.25
        
        # Identificação de melhorias
        if any(word in reflection_lower for word in ['melhorar', 'melhoria', 'refinamento', 'ajustar']):
            score += 0.25
        
        # Estrutura organizada
        if any(pattern in reflection_lower for pattern in ['1.', '2.', '3.', 'primeiro', 'segundo']):
            score += 0.25
        
        return score
    
    # Retornar os métodos para serem adicionados à classe
    return {
        'agentic_multiagent_evaluation': agentic_multiagent_evaluation,
        '_run_agent_with_tools': _run_agent_with_tools,
        '_extract_final_answer_from_synthesis': _extract_final_answer_from_synthesis,
        '_extract_confidence_from_verification': _extract_confidence_from_verification,
        'self_reflection_evaluation': self_reflection_evaluation,
        '_evaluate_reflection_quality': _evaluate_reflection_quality
    }

def add_agentic_methods_to_evaluator():
    """
    Adiciona métodos agenticos à classe ScientificEvaluator existente
    """
    
    def comprehensive_evaluation_with_agentic(self, models_dict, datasets):
        """
        Avaliação científica incluindo técnicas Agentic AI
        """
        print("🤖 Iniciando Avaliação Científica com Agentic AI...")
        
        # Primeiro, executar avaliação padrão se disponível
        if hasattr(self, 'comprehensive_evaluation'):
            results_matrix, detailed_results = self.comprehensive_evaluation(models_dict, datasets)
        else:
            results_matrix, detailed_results = {}, {}
        
        # Adicionar avaliações Agentic AI
        agentic_strategies = {
            'Agentic Multi-Agent': 'agentic_multiagent_evaluation',
            'Agentic Self-Reflection': 'self_reflection_evaluation'
        }
        
        for model_name, (model, tokenizer) in models_dict.items():
            print(f"\n🤖 Avaliações Agentic para modelo: {model_name}")
            model.to(self.config.DEVICE)
            
            # Criar instância do prompt engineer agentico
            if not hasattr(self, 'agentic_prompt_engineer'):
                # Assumir que PromptEngineer já foi estendido com métodos agenticos
                self.agentic_prompt_engineer = type('MockPromptEngineer', (), {})()
                self.agentic_prompt_engineer.config = self.config
                self.agentic_prompt_engineer.logger = self.logger
                
                # Adicionar métodos agenticos
                agentic_methods = add_agentic_methods_to_prompt_engineer()
                for method_name, method_func in agentic_methods.items():
                    setattr(self.agentic_prompt_engineer, method_name, 
                           method_func.__get__(self.agentic_prompt_engineer))
            
            for strategy_name, method_name in agentic_strategies.items():
                print(f"  🔬 Estratégia: {strategy_name}")
                
                try:
                    eval_method = getattr(self.agentic_prompt_engineer, method_name)
                    results = eval_method(model, tokenizer, datasets['evaluation'], datasets['metrics_calculator'])
                    
                    # Adicionar aos resultados existentes
                    if model_name not in results_matrix:
                        results_matrix[model_name] = {}
                    if model_name not in detailed_results:
                        detailed_results[model_name] = {}
                    
                    results_matrix[model_name][strategy_name] = {
                        'f1': results['f1'],
                        'exact_match': results['exact_match']
                    }
                    detailed_results[model_name][strategy_name] = results
                    
                    print(f"    ✅ F1: {results['f1']:.2f}, EM: {results['exact_match']:.2f}")
                    
                    # Log métricas específicas
                    if 'avg_confidence' in results:
                        print(f"    🎯 Confiança: {results['avg_confidence']:.2f}")
                    if 'acceptance_rate' in results:
                        print(f"    ✓ Taxa de aceitação: {results['acceptance_rate']*100:.1f}%")
                    if 'avg_reflection_quality' in results:
                        print(f"    🧠 Qualidade reflexão: {results['avg_reflection_quality']:.2f}")
                    
                except Exception as e:
                    print(f"    ❌ Erro: {e}")
                    results_matrix[model_name][strategy_name] = {'f1': 0.0, 'exact_match': 0.0}
                    detailed_results[model_name][strategy_name] = {'error': str(e)}
        
        return results_matrix, detailed_results
    
    return {
        'comprehensive_evaluation_with_agentic': comprehensive_evaluation_with_agentic
    }

# FUNÇÃO DE INTEGRAÇÃO PRINCIPAL
def integrate_agentic_workflow():
    """
    Função principal para integrar AgenticWorkflow com SLM_Final
    
    Como usar:
    1. Execute todas as células do SLM_Final notebook
    2. Execute: exec(open('AgenticWorkflow_Integration.py').read())
    3. Execute: integrate_agentic_workflow()
    4. As classes PromptEngineer e ScientificEvaluator terão novos métodos agenticos
    """
    
    print("🚀 INTEGRANDO AGENTIC WORKFLOW COM SLM_FINAL")
    print("="*60)
    
    try:
        # Verificar se as classes base existem
        if 'PromptEngineer' in globals():
            print("✅ PromptEngineer encontrado - adicionando métodos agenticos...")
            
            # Adicionar métodos agenticos à classe PromptEngineer
            agentic_methods = add_agentic_methods_to_prompt_engineer()
            for method_name, method_func in agentic_methods.items():
                setattr(PromptEngineer, method_name, method_func)
            
            print("   ✓ agentic_multiagent_evaluation")
            print("   ✓ self_reflection_evaluation")
            print("   ✓ métodos auxiliares")
            
        else:
            print("⚠️  PromptEngineer não encontrado - certifique-se de executar as células do SLM_Final primeiro")
        
        if 'ScientificEvaluator' in globals():
            print("✅ ScientificEvaluator encontrado - adicionando métodos agenticos...")
            
            # Adicionar métodos agenticos à classe ScientificEvaluator
            evaluator_methods = add_agentic_methods_to_evaluator()
            for method_name, method_func in evaluator_methods.items():
                setattr(ScientificEvaluator, method_name, method_func)
            
            print("   ✓ comprehensive_evaluation_with_agentic")
            
        else:
            print("⚠️  ScientificEvaluator não encontrado - certifique-se de executar as células do SLM_Final primeiro")
        
        print("\n🎯 COMO USAR OS NOVOS MÉTODOS:")
        print("# Para avaliação multiagente:")
        print("prompt_engineer = PromptEngineer(config, logger)")
        print("results = prompt_engineer.agentic_multiagent_evaluation(model, tokenizer, eval_dataset, metrics_calc)")
        print("")
        print("# Para auto-reflexão:")
        print("results = prompt_engineer.self_reflection_evaluation(model, tokenizer, eval_dataset, metrics_calc)")
        print("")
        print("# Para avaliação completa com agentic:")
        print("evaluator = ScientificEvaluator(config, logger)")
        print("results_matrix, detailed = evaluator.comprehensive_evaluation_with_agentic(models_dict, datasets)")
        
        print("\n✅ INTEGRAÇÃO CONCLUÍDA COM SUCESSO!")
        return True
        
    except Exception as e:
        print(f"❌ Erro na integração: {e}")
        return False

# FUNÇÃO DE DEMONSTRAÇÃO RÁPIDA
def quick_agentic_demo():
    """
    Demonstração rápida das capacidades agenticas
    Executa apenas se as classes do SLM_Final estiverem disponíveis
    """
    print("🎯 DEMONSTRAÇÃO RÁPIDA AGENTIC AI")
    print("="*50)
    
    try:
        # Verificar disponibilidade das classes necessárias
        required_classes = ['ScientificExperimentRunner', 'PromptEngineer', 'ScientificEvaluator']
        missing_classes = [cls for cls in required_classes if cls not in globals()]
        
        if missing_classes:
            print(f"❌ Classes necessárias não encontradas: {missing_classes}")
            print("Execute primeiro todas as células do SLM_Final notebook")
            return None
        
        # Integrar métodos agenticos
        integrate_agentic_workflow()
        
        # Setup básico
        runner = ScientificExperimentRunner()
        config = runner.config
        logger = runner.logger
        
        # Configurações reduzidas para demo
        config.EVAL_SUBSET_SIZE = 5
        config.MAX_LENGTH = 256
        
        print("\n📊 Carregando dados para demonstração...")
        datasets, _ = runner.dataset_manager.load_and_analyze_data()
        
        print("🤖 Preparando modelo...")
        models_dict = runner._prepare_evaluation_models({})
        
        if not models_dict:
            print("⚡ Carregando modelo base...")
            base_model, base_tokenizer = runner._load_model_safe(config.STUDENT_MODEL_ID)
            if base_model:
                models_dict['SLM Base'] = (base_model, base_tokenizer)
        
        if not models_dict:
            print("❌ Não foi possível carregar modelo")
            return None
        
        print(f"✅ Modelo carregado: {list(models_dict.keys())}")
        
        # Demonstração com PromptEngineer
        print(f"\n{'='*40}")
        print("🤖 DEMO: MULTIAGENT EVALUATION")
        print(f"{'='*40}")
        
        prompt_engineer = PromptEngineer(config, logger)
        model_name, (model, tokenizer) = next(iter(models_dict.items()))
        
        results = prompt_engineer.agentic_multiagent_evaluation(
            model, tokenizer, 
            datasets['evaluation'].select(range(3)), 
            datasets['metrics_calculator']
        )
        
        print(f"\n📊 RESULTADOS DEMO:")
        print(f"F1-Score: {results['f1']:.3f}")
        print(f"Exact Match: {results['exact_match']:.3f}")
        if 'avg_confidence' in results:
            print(f"Confiança média: {results['avg_confidence']:.3f}")
        if 'acceptance_rate' in results:
            print(f"Taxa de aceitação: {results['acceptance_rate']*100:.1f}%")
        
        print("\n✅ Demonstração concluída!")
        return results
        
    except Exception as e:
        print(f"❌ Erro na demonstração: {e}")
        import traceback
        traceback.print_exc()
        return None

In [ ]:
print("🔬 INICIALIZANDO EXPERIMENTOS SISTEMÁTICOS")
print("="*60)

# Initialize results collection
systematic_results = {
    'experiment_metadata': {
        'timestamp': datetime.now().isoformat(),
        'total_combinations': 0,
        'completed_experiments': 0
    },
    'results_matrix': {},
    'detailed_results': {},
    'comparison_data': []
}

# Define experimental combinations
KD_TECHNIQUES = {
    'No_KD': 'Baseline sem Knowledge Distillation',
    'Logits_KD': 'Knowledge Distillation baseado em Logits', 
    'Self_Distillation': 'Auto-destilação progressiva',
    'Progressive_KD': 'Destilação progressiva multi-estágio'
}

PROMPT_STRATEGIES = {
    'Zero_Shot': 'Zero-Shot direto',
    'Few_Shot': '3-Shot In-Context Learning',
    'Chain_of_Thought': 'Zero-Shot Chain-of-Thought',
    'Self_Consistency': 'Self-Consistency com múltiplas amostras',
    'Agentic_MultiAgent': 'Framework Multiagente Agentic'
}

total_combinations = len(KD_TECHNIQUES) * len(PROMPT_STRATEGIES)
systematic_results['experiment_metadata']['total_combinations'] = total_combinations

print(f"📊 {len(KD_TECHNIQUES)} técnicas de KD × {len(PROMPT_STRATEGIES)} estratégias de prompt = {total_combinations} experimentos")
print(f"🎯 Cada experimento será executado e os resultados coletados sistematicamente")

# Ensure results directory exists
os.makedirs('./systematic_results', exist_ok=True)

print("✅ Inicialização completa. Execute as células seguintes para cada experimento.")

CONFIGURAÇÃO DO EXPERIMENTO
Modelo Estudante: google/gemma-2b
Modelo Professor: microsoft/DialoGPT-medium
Dispositivo: cuda
Tamanho Avaliação: 100
Treinamento: False
Avaliação: ✅
INICIANDO EXPERIMENTO CIENTÍFICO COMPLETO
Carregando e analisando datasets...


README.md: 0.00B [00:00, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

validation-00000-of-00001.parquet:   0%|          | 0.00/1.82M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

...

    ✅ F1: 1.08, EM: 0.00
  📋 Estratégia: Self-Consistency
🎯 Auto-Consistência com 5 caminhos...


Self-Consistency:   0%|          | 0/50 [00:00<?, ?it/s]

    ✅ F1: 3.91, EM: 0.00
  📋 Estratégia: Workflow Agente-Executor
🔍 Avaliando: Workflow Agente-Executor


Workflow Agente-Executor:   0%|          | 0/100 [00:00<?, ?it/s]

Erro no workflow do agente: 'PromptEngineer' object has no attribute '_create_analyst_prompt'
Erro no workflow do agente: 'PromptEngineer' object has no attribute '_create_analyst_prompt'
Erro no workflow do agente: 'PromptEngineer' object has no attribute '_create_analyst_prompt'
Erro no workflow do agente: 'PromptEngineer' object has no attribute '_create_analyst_prompt'
Erro no workflow do agente: 'PromptEngineer' object has no attribute '_create_analyst_prompt'
Erro no workflow do agente: 'PromptEngineer' object has no attribute '_create_analyst_prompt'
Erro no workflow do agente: 'PromptEngineer' object has no attribute '_create_analyst_prompt'
Erro no workflow do agente: 'PromptEngineer' object has no attribute '_create_analyst_prompt'
Erro no workflow do agente: 'PromptEngineer' object has no attribute '_create_analyst_prompt'
Erro no workflow do agente: 'PromptEngineer' object has no attribute '_create_analyst_prompt'
Erro no workflow do agente: 'PromptEngineer' object has no a

In [ ]:
def run_experiment_no_kd_zero_shot():
    """Experimento 1: Baseline sem KD + Zero-Shot"""
    
    experiment_name = "No_KD_Zero_Shot"
    print(f"🧪 EXPERIMENTO 1: {experiment_name}")
    print("="*50)
    print("📋 Configuração: Modelo base + Prompt Zero-Shot")
    
    try:
        # Usar modelo base (sem KD)
        if 'runner' in globals() and hasattr(runner, 'config'):
            config = runner.config
            logger = runner.logger
        else:
            print("❌ Runner não encontrado. Execute primeiro as células do SLM_Final.")
            return None
            
        # Carregar dados se necessário
        if 'datasets' not in globals():
            datasets, _ = runner.dataset_manager.load_and_analyze_data()
        
        # Preparar modelo base
        base_model, base_tokenizer = runner._load_model_safe(config.STUDENT_MODEL_ID)
        if not base_model:
            print("❌ Falha ao carregar modelo base")
            return None
            
        print("✅ Modelo base carregado")
        
        # Executar avaliação Zero-Shot
        prompt_engineer = PromptEngineer(config, logger)
        
        results = prompt_engineer.evaluate_with_prompt_strategy(
            base_model, base_tokenizer, 
            datasets['evaluation'].select(range(50)),  # Subset para eficiência
            datasets['metrics_calculator'],
            "Zero-Shot",
            prompt_engineer.zero_shot_prompt
        )
        
        # Armazenar resultados
        experiment_result = {
            'kd_technique': 'No_KD',
            'prompt_strategy': 'Zero_Shot', 
            'f1_score': results['f1'],
            'exact_match': results['exact_match'],
            'model_info': 'Base model without KD',
            'experiment_details': results
        }
        
        systematic_results['results_matrix'][experiment_name] = experiment_result
        systematic_results['comparison_data'].append(experiment_result)
        systematic_results['experiment_metadata']['completed_experiments'] += 1
        
        print(f"📊 RESULTADOS - {experiment_name}:")
        print(f"   F1-Score: {results['f1']:.3f}")
        print(f"   Exact Match: {results['exact_match']:.3f}")
        print("✅ Experimento 1 concluído")
        
        return experiment_result
        
    except Exception as e:
        print(f"❌ Erro no Experimento 1: {e}")
        return None

# Execute experiment 1
exp1_result = run_experiment_no_kd_zero_shot()


In [ ]:
def run_experiment_no_kd_few_shot():
    """Experimento 2: Baseline sem KD + Few-Shot"""
    
    experiment_name = "No_KD_Few_Shot"
    print(f"🧪 EXPERIMENTO 2: {experiment_name}")
    print("="*50)
    print("📋 Configuração: Modelo base + 3-Shot In-Context Learning")
    
    try:
        # Usar modelo base (sem KD) 
        if 'runner' not in globals():
            print("❌ Runner não encontrado")
            return None
            
        config = runner.config
        logger = runner.logger
        
        # Carregar dados
        if 'datasets' not in globals():
            datasets, _ = runner.dataset_manager.load_and_analyze_data()
        
        # Preparar modelo base
        base_model, base_tokenizer = runner._load_model_safe(config.STUDENT_MODEL_ID)
        if not base_model:
            print("❌ Falha ao carregar modelo base")
            return None
            
        print("✅ Modelo base carregado")
        
        # Executar avaliação Few-Shot
        prompt_engineer = PromptEngineer(config, logger)
        
        results = prompt_engineer.evaluate_with_prompt_strategy(
            base_model, base_tokenizer,
            datasets['evaluation'].select(range(50)),
            datasets['metrics_calculator'],
            "3-Shot ICL",
            prompt_engineer.few_shot_prompt,
            icl_examples=datasets['icl_examples'],
            num_shots=3
        )
        
        # Armazenar resultados
        experiment_result = {
            'kd_technique': 'No_KD',
            'prompt_strategy': 'Few_Shot',
            'f1_score': results['f1'],
            'exact_match': results['exact_match'],
            'model_info': 'Base model without KD',
            'experiment_details': results
        }
        
        systematic_results['results_matrix'][experiment_name] = experiment_result
        systematic_results['comparison_data'].append(experiment_result)
        systematic_results['experiment_metadata']['completed_experiments'] += 1
        
        print(f"📊 RESULTADOS - {experiment_name}:")
        print(f"   F1-Score: {results['f1']:.3f}")
        print(f"   Exact Match: {results['exact_match']:.3f}")
        print("✅ Experimento 2 concluído")
        
        return experiment_result
        
    except Exception as e:
        print(f"❌ Erro no Experimento 2: {e}")
        return None

# Execute experiment 2
exp2_result = run_experiment_no_kd_few_shot()

In [ ]:
def run_experiment_no_kd_cot():
    """Experimento 3: Baseline sem KD + Chain-of-Thought"""
    
    experiment_name = "No_KD_CoT"
    print(f"🧪 EXPERIMENTO 3: {experiment_name}")
    print("="*50)
    print("📋 Configuração: Modelo base + Zero-Shot Chain-of-Thought")
    
    try:
        if 'runner' not in globals():
            print("❌ Runner não encontrado")
            return None
            
        config = runner.config
        logger = runner.logger
        
        if 'datasets' not in globals():
            datasets, _ = runner.dataset_manager.load_and_analyze_data()
        
        # Preparar modelo base
        base_model, base_tokenizer = runner._load_model_safe(config.STUDENT_MODEL_ID)
        if not base_model:
            print("❌ Falha ao carregar modelo base")
            return None
            
        print("✅ Modelo base carregado")
        
        # Executar avaliação Chain-of-Thought
        prompt_engineer = PromptEngineer(config, logger)
        
        results = prompt_engineer.evaluate_with_prompt_strategy(
            base_model, base_tokenizer,
            datasets['evaluation'].select(range(50)),
            datasets['metrics_calculator'],
            "Zero-Shot CoT",
            prompt_engineer.cot_prompt
        )
        
        # Armazenar resultados
        experiment_result = {
            'kd_technique': 'No_KD',
            'prompt_strategy': 'Chain_of_Thought',
            'f1_score': results['f1'],
            'exact_match': results['exact_match'],
            'model_info': 'Base model without KD',
            'experiment_details': results
        }
        
        systematic_results['results_matrix'][experiment_name] = experiment_result
        systematic_results['comparison_data'].append(experiment_result)
        systematic_results['experiment_metadata']['completed_experiments'] += 1
        
        print(f"📊 RESULTADOS - {experiment_name}:")
        print(f"   F1-Score: {results['f1']:.3f}")
        print(f"   Exact Match: {results['exact_match']:.3f}")
        print("✅ Experimento 3 concluído")
        
        return experiment_result
        
    except Exception as e:
        print(f"❌ Erro no Experimento 3: {e}")
        return None

# Execute experiment 3
exp3_result = run_experiment_no_kd_cot()

In [ ]:
def run_experiment_no_kd_self_consistency():
    """Experimento 4: Baseline sem KD + Self-Consistency"""
    
    experiment_name = "No_KD_Self_Consistency"
    print(f"🧪 EXPERIMENTO 4: {experiment_name}")
    print("="*50)
    print("📋 Configuração: Modelo base + Self-Consistency")
    
    try:
        if 'runner' not in globals():
            print("❌ Runner não encontrado")
            return None
            
        config = runner.config
        logger = runner.logger
        
        if 'datasets' not in globals():
            datasets, _ = runner.dataset_manager.load_and_analyze_data()
        
        # Preparar modelo base
        base_model, base_tokenizer = runner._load_model_safe(config.STUDENT_MODEL_ID)
        if not base_model:
            print("❌ Falha ao carregar modelo base")
            return None
            
        print("✅ Modelo base carregado")
        
        # Executar avaliação Self-Consistency
        prompt_engineer = PromptEngineer(config, logger)
        
        results = prompt_engineer.self_consistency_evaluation(
            base_model, base_tokenizer,
            datasets['evaluation'].select(range(30)),  # Subset menor - Self-Consistency é mais lento
            datasets['metrics_calculator']
        )
        
        # Armazenar resultados
        experiment_result = {
            'kd_technique': 'No_KD',
            'prompt_strategy': 'Self_Consistency',
            'f1_score': results['f1'],
            'exact_match': results['exact_match'],
            'model_info': 'Base model without KD',
            'experiment_details': results
        }
        
        systematic_results['results_matrix'][experiment_name] = experiment_result
        systematic_results['comparison_data'].append(experiment_result)
        systematic_results['experiment_metadata']['completed_experiments'] += 1
        
        print(f"📊 RESULTADOS - {experiment_name}:")
        print(f"   F1-Score: {results['f1']:.3f}")
        print(f"   Exact Match: {results['exact_match']:.3f}")
        if 'consistency_score' in results:
            print(f"   Consistency Score: {results['consistency_score']:.3f}")
        print("✅ Experimento 4 concluído")
        
        return experiment_result
        
    except Exception as e:
        print(f"❌ Erro no Experimento 4: {e}")
        return None

# Execute experiment 4
exp4_result = run_experiment_no_kd_self_consistency()

In [ ]:
def run_experiment_no_kd_agentic():
    """Experimento 5: Baseline sem KD + Agentic Multi-Agent"""
    
    experiment_name = "No_KD_Agentic"
    print(f"🧪 EXPERIMENTO 5: {experiment_name}")
    print("="*50)
    print("📋 Configuração: Modelo base + Framework Multiagente Agentic")
    
    try:
        if 'runner' not in globals():
            print("❌ Runner não encontrado")
            return None
            
        config = runner.config
        logger = runner.logger
        
        if 'datasets' not in globals():
            datasets, _ = runner.dataset_manager.load_and_analyze_data()
        
        # Preparar modelo base
        base_model, base_tokenizer = runner._load_model_safe(config.STUDENT_MODEL_ID)
        if not base_model:
            print("❌ Falha ao carregar modelo base")
            return None
            
        print("✅ Modelo base carregado")
        
        # Carregar integração Agentic se não estiver disponível
        if not hasattr(PromptEngineer, 'agentic_multiagent_evaluation'):
            print("🔧 Carregando integração Agentic...")
            exec(open('AgenticWorkflow_Integration.py').read())
            integrate_agentic_workflow()
        
        # Executar avaliação Agentic
        prompt_engineer = PromptEngineer(config, logger)
        
        results = prompt_engineer.agentic_multiagent_evaluation(
            base_model, base_tokenizer,
            datasets['evaluation'].select(range(25)),  # Subset menor - Agentic é mais lento
            datasets['metrics_calculator']
        )
        
        # Armazenar resultados
        experiment_result = {
            'kd_technique': 'No_KD',
            'prompt_strategy': 'Agentic_MultiAgent',
            'f1_score': results['f1'],
            'exact_match': results['exact_match'],
            'model_info': 'Base model without KD',
            'experiment_details': results
        }
        
        # Adicionar métricas específicas do Agentic
        if 'avg_confidence' in results:
            experiment_result['avg_confidence'] = results['avg_confidence']
        if 'acceptance_rate' in results:
            experiment_result['acceptance_rate'] = results['acceptance_rate']
        
        systematic_results['results_matrix'][experiment_name] = experiment_result
        systematic_results['comparison_data'].append(experiment_result)
        systematic_results['experiment_metadata']['completed_experiments'] += 1
        
        print(f"📊 RESULTADOS - {experiment_name}:")
        print(f"   F1-Score: {results['f1']:.3f}")
        print(f"   Exact Match: {results['exact_match']:.3f}")
        if 'avg_confidence' in results:
            print(f"   Avg Confidence: {results['avg_confidence']:.3f}")
        if 'acceptance_rate' in results:
            print(f"   Acceptance Rate: {results['acceptance_rate']*100:.1f}%")
        print("✅ Experimento 5 concluído")
        
        return experiment_result
        
    except Exception as e:
        print(f"❌ Erro no Experimento 5: {e}")
        return None

# Execute experiment 5
exp5_result = run_experiment_no_kd_agentic()

In [ ]:
def run_experiment_logits_kd_zero_shot():
    """Experimento 6: Logits KD + Zero-Shot"""
    
    experiment_name = "Logits_KD_Zero_Shot"
    print(f"🧪 EXPERIMENTO 6: {experiment_name}")
    print("="*50)
    print("📋 Configuração: Modelo com Logits KD + Zero-Shot")
    
    try:
        if 'runner' not in globals():
            print("❌ Runner não encontrado")
            return None
            
        config = runner.config
        logger = runner.logger
        
        if 'datasets' not in globals():
            datasets, _ = runner.dataset_manager.load_and_analyze_data()
        
        # Preparar modelos para KD
        teacher_model, teacher_tokenizer = runner._load_model_safe(config.TEACHER_MODEL_ID)
        student_model, student_tokenizer = runner._load_model_safe(config.STUDENT_MODEL_ID)
        
        if not teacher_model or not student_model:
            print("❌ Falha ao carregar modelos para KD")
            return None
            
        print("✅ Modelos carregados para Knowledge Distillation")
        
        # Executar Knowledge Distillation
        distiller = KnowledgeDistiller(config, logger)
        
        kd_model, kd_metrics = distiller.distill_with_logits(
            student_model, teacher_model,
            student_tokenizer, teacher_tokenizer,
            datasets['kd_transfer'].select(range(100)),  # Subset para KD
            "Logits_KD_Experiment"
        )
        
        print("✅ Knowledge Distillation concluído")
        
        # Executar avaliação Zero-Shot no modelo destilado
        prompt_engineer = PromptEngineer(config, logger)
        
        results = prompt_engineer.evaluate_with_prompt_strategy(
            kd_model, student_tokenizer,
            datasets['evaluation'].select(range(50)),
            datasets['metrics_calculator'],
            "Zero-Shot",
            prompt_engineer.zero_shot_prompt
        )
        
        # Armazenar resultados
        experiment_result = {
            'kd_technique': 'Logits_KD',
            'prompt_strategy': 'Zero_Shot',
            'f1_score': results['f1'],
            'exact_match': results['exact_match'],
            'model_info': 'Model trained with Logits Knowledge Distillation',
            'kd_metrics': kd_metrics,
            'experiment_details': results
        }
        
        systematic_results['results_matrix'][experiment_name] = experiment_result
        systematic_results['comparison_data'].append(experiment_result)
        systematic_results['experiment_metadata']['completed_experiments'] += 1
        
        print(f"📊 RESULTADOS - {experiment_name}:")
        print(f"   F1-Score: {results['f1']:.3f}")
        print(f"   Exact Match: {results['exact_match']:.3f}")
        print(f"   KD Training Loss: {kd_metrics.get('final_loss', 'N/A')}")
        print("✅ Experimento 6 concluído")
        
        return experiment_result
        
    except Exception as e:
        print(f"❌ Erro no Experimento 6: {e}")
        return None

# Execute experiment 6
exp6_result = run_experiment_logits_kd_zero_shot()

In [ ]:
def run_experiment_logits_kd_few_shot():
    """Experimento 7: Logits KD + Few-Shot"""
    
    experiment_name = "Logits_KD_Few_Shot"
    print(f"🧪 EXPERIMENTO 7: {experiment_name}")
    print("="*50)
    print("📋 Configuração: Modelo com Logits KD + 3-Shot ICL")
    
    try:
        if 'runner' not in globals():
            print("❌ Runner não encontrado")
            return None
            
        config = runner.config
        logger = runner.logger
        
        if 'datasets' not in globals():
            datasets, _ = runner.dataset_manager.load_and_analyze_data()
        
        # Preparar modelos para KD
        teacher_model, teacher_tokenizer = runner._load_model_safe(config.TEACHER_MODEL_ID)
        student_model, student_tokenizer = runner._load_model_safe(config.STUDENT_MODEL_ID)
        
        if not teacher_model or not student_model:
            print("❌ Falha ao carregar modelos para KD")
            return None
            
        print("✅ Modelos carregados para Knowledge Distillation")
        
        # Executar Knowledge Distillation
        distiller = KnowledgeDistiller(config, logger)
        
        kd_model, kd_metrics = distiller.distill_with_logits(
            student_model, teacher_model,
            student_tokenizer, teacher_tokenizer,
            datasets['kd_transfer'].select(range(100)),
            "Logits_KD_FewShot_Experiment"
        )
        
        print("✅ Knowledge Distillation concluído")
        
        # Executar avaliação Few-Shot no modelo destilado
        prompt_engineer = PromptEngineer(config, logger)
        
        results = prompt_engineer.evaluate_with_prompt_strategy(
            kd_model, student_tokenizer,
            datasets['evaluation'].select(range(50)),
            datasets['metrics_calculator'],
            "3-Shot ICL",
            prompt_engineer.few_shot_prompt,
            icl_examples=datasets['icl_examples'],
            num_shots=3
        )
        
        # Armazenar resultados
        experiment_result = {
            'kd_technique': 'Logits_KD',
            'prompt_strategy': 'Few_Shot',
            'f1_score': results['f1'],
            'exact_match': results['exact_match'],
            'model_info': 'Model trained with Logits Knowledge Distillation',
            'kd_metrics': kd_metrics,
            'experiment_details': results
        }
        
        systematic_results['results_matrix'][experiment_name] = experiment_result
        systematic_results['comparison_data'].append(experiment_result)
        systematic_results['experiment_metadata']['completed_experiments'] += 1
        
        print(f"📊 RESULTADOS - {experiment_name}:")
        print(f"   F1-Score: {results['f1']:.3f}")
        print(f"   Exact Match: {results['exact_match']:.3f}")
        print(f"   KD Training Loss: {kd_metrics.get('final_loss', 'N/A')}")
        print("✅ Experimento 7 concluído")
        
        return experiment_result
        
    except Exception as e:
        print(f"❌ Erro no Experimento 7: {e}")
        return None

# Execute experiment 7
exp7_result = run_experiment_logits_kd_few_shot()

In [ ]:
def run_experiment_self_distillation_cot():
    """Experimento 8: Self-Distillation + Chain-of-Thought"""
    
    experiment_name = "Self_Distillation_CoT"
    print(f"🧪 EXPERIMENTO 8: {experiment_name}")
    print("="*50)
    print("📋 Configuração: Modelo com Auto-Destilação + Chain-of-Thought")
    
    try:
        if 'runner' not in globals():
            print("❌ Runner não encontrado")
            return None
            
        config = runner.config
        logger = runner.logger
        
        if 'datasets' not in globals():
            datasets, _ = runner.dataset_manager.load_and_analyze_data()
        
        # Preparar modelo para auto-destilação
        base_model, base_tokenizer = runner._load_model_safe(config.STUDENT_MODEL_ID)
        if not base_model:
            print("❌ Falha ao carregar modelo base")
            return None
            
        print("✅ Modelo base carregado para auto-destilação")
        
        # Executar Self-Distillation
        distiller = KnowledgeDistiller(config, logger)
        
        self_distilled_model, sd_metrics = distiller.self_distillation(
            base_model, base_tokenizer,
            datasets['kd_transfer'].select(range(100))
        )
        
        print("✅ Auto-Destilação concluída")
        
        # Executar avaliação Chain-of-Thought no modelo auto-destilado
        prompt_engineer = PromptEngineer(config, logger)
        
        results = prompt_engineer.evaluate_with_prompt_strategy(
            self_distilled_model, base_tokenizer,
            datasets['evaluation'].select(range(50)),
            datasets['metrics_calculator'],
            "Zero-Shot CoT",
            prompt_engineer.cot_prompt
        )
        
        # Armazenar resultados
        experiment_result = {
            'kd_technique': 'Self_Distillation',
            'prompt_strategy': 'Chain_of_Thought',
            'f1_score': results['f1'],
            'exact_match': results['exact_match'],
            'model_info': 'Model trained with Self-Distillation',
            'kd_metrics': sd_metrics,
            'experiment_details': results
        }
        
        systematic_results['results_matrix'][experiment_name] = experiment_result
        systematic_results['comparison_data'].append(experiment_result)
        systematic_results['experiment_metadata']['completed_experiments'] += 1
        
        print(f"📊 RESULTADOS - {experiment_name}:")
        print(f"   F1-Score: {results['f1']:.3f}")
        print(f"   Exact Match: {results['exact_match']:.3f}")
        print(f"   Self-Distillation Loss: {sd_metrics.get('final_loss', 'N/A')}")
        print("✅ Experimento 8 concluído")
        
        return experiment_result
        
    except Exception as e:
        print(f"❌ Erro no Experimento 8: {e}")
        return None

# Execute experiment 8
exp8_result = run_experiment_self_distillation_cot()

In [ ]:
def run_experiment_self_distillation_agentic():
    """Experimento 9: Self-Distillation + Agentic Multi-Agent"""
    
    experiment_name = "Self_Distillation_Agentic"
    print(f"🧪 EXPERIMENTO 9: {experiment_name}")
    print("="*50)
    print("📋 Configuração: Modelo com Auto-Destilação + Framework Agentic")
    
    try:
        if 'runner' not in globals():
            print("❌ Runner não encontrado")
            return None
            
        config = runner.config
        logger = runner.logger
        
        if 'datasets' not in globals():
            datasets, _ = runner.dataset_manager.load_and_analyze_data()
        
        # Preparar modelo para auto-destilação
        base_model, base_tokenizer = runner._load_model_safe(config.STUDENT_MODEL_ID)
        if not base_model:
            print("❌ Falha ao carregar modelo base")
            return None
            
        print("✅ Modelo base carregado para auto-destilação")
        
        # Executar Self-Distillation
        distiller = KnowledgeDistiller(config, logger)
        
        self_distilled_model, sd_metrics = distiller.self_distillation(
            base_model, base_tokenizer,
            datasets['kd_transfer'].select(range(100))
        )
        
        print("✅ Auto-Destilação concluída")
        
        # Carregar integração Agentic se não estiver disponível
        if not hasattr(PromptEngineer, 'agentic_multiagent_evaluation'):
            print("🔧 Carregando integração Agentic...")
            exec(open('AgenticWorkflow_Integration.py').read())
            integrate_agentic_workflow()
        
        # Executar avaliação Agentic no modelo auto-destilado
        prompt_engineer = PromptEngineer(config, logger)
        
        results = prompt_engineer.agentic_multiagent_evaluation(
            self_distilled_model, base_tokenizer,
            datasets['evaluation'].select(range(25)),
            datasets['metrics_calculator']
        )
        
        # Armazenar resultados
        experiment_result = {
            'kd_technique': 'Self_Distillation',
            'prompt_strategy': 'Agentic_MultiAgent',
            'f1_score': results['f1'],
            'exact_match': results['exact_match'],
            'model_info': 'Model trained with Self-Distillation',
            'kd_metrics': sd_metrics,
            'experiment_details': results
        }
        
        # Adicionar métricas específicas do Agentic
        if 'avg_confidence' in results:
            experiment_result['avg_confidence'] = results['avg_confidence']
        if 'acceptance_rate' in results:
            experiment_result['acceptance_rate'] = results['acceptance_rate']
        
        systematic_results['results_matrix'][experiment_name] = experiment_result
        systematic_results['comparison_data'].append(experiment_result)
        systematic_results['experiment_metadata']['completed_experiments'] += 1
        
        print(f"📊 RESULTADOS - {experiment_name}:")
        print(f"   F1-Score: {results['f1']:.3f}")
        print(f"   Exact Match: {results['exact_match']:.3f}")
        if 'avg_confidence' in results:
            print(f"   Avg Confidence: {results['avg_confidence']:.3f}")
        if 'acceptance_rate' in results:
            print(f"   Acceptance Rate: {results['acceptance_rate']*100:.1f}%")
        print(f"   Self-Distillation Loss: {sd_metrics.get('final_loss', 'N/A')}")
        print("✅ Experimento 9 concluído")
        
        return experiment_result
        
    except Exception as e:
        print(f"❌ Erro no Experimento 9: {e}")
        return None

# Execute experiment 9
exp9_result = run_experiment_self_distillation_agentic()

In [ ]:
def collect_and_analyze_results():
    """Coleta e analisa todos os resultados dos experimentos sistemáticos"""
    
    print("📊 COLETANDO E ANALISANDO RESULTADOS SISTEMÁTICOS")
    print("="*60)
    
    # Atualizar metadata
    systematic_results['experiment_metadata']['analysis_timestamp'] = datetime.now().isoformat()
    
    # Criar DataFrame para análise
    df_results = pd.DataFrame(systematic_results['comparison_data'])
    
    if len(df_results) == 0:
        print("❌ Nenhum resultado encontrado para análise")
        return None
    
    print(f"✅ Analisando {len(df_results)} experimentos concluídos")
    
    # Análise por técnica de KD
    print(f"\n🔬 ANÁLISE POR TÉCNICA DE KNOWLEDGE DISTILLATION:")
    kd_analysis = df_results.groupby('kd_technique')[['f1_score', 'exact_match']].mean()
    print(kd_analysis.round(3))
    
    # Análise por estratégia de prompt
    print(f"\n💭 ANÁLISE POR ESTRATÉGIA DE PROMPT:")
    prompt_analysis = df_results.groupby('prompt_strategy')[['f1_score', 'exact_match']].mean()
    print(prompt_analysis.round(3))
    
    # Encontrar melhor combinação
    best_experiment = df_results.loc[df_results['f1_score'].idxmax()]
    print(f"\n🏆 MELHOR COMBINAÇÃO:")
    print(f"   Técnica KD: {best_experiment['kd_technique']}")
    print(f"   Estratégia Prompt: {best_experiment['prompt_strategy']}")
    print(f"   F1-Score: {best_experiment['f1_score']:.3f}")
    print(f"   Exact Match: {best_experiment['exact_match']:.3f}")
    
    # Salvar resultados completos
    results_file = f"./systematic_results/systematic_experiments_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
    
    with open(results_file, 'w') as f:
        json.dump(systematic_results, f, indent=2, default=str)
    
    print(f"\n💾 Resultados salvos em: {results_file}")
    
    # Criar tabela resumo
    pivot_table = df_results.pivot_table(
        values='f1_score', 
        index='kd_technique', 
        columns='prompt_strategy', 
        fill_value=0
    )
    
    print(f"\n📋 TABELA RESUMO - F1 SCORES:")
    print(pivot_table.round(3))
    
    return df_results, pivot_table

# Execute results collection
df_systematic, pivot_systematic = collect_and_analyze_results()

In [ ]:
def visualize_systematic_results(df_results, pivot_table):
    """Cria visualizações dos resultados sistemáticos"""
    
    import matplotlib.pyplot as plt
    import seaborn as sns
    
    print("📈 CRIANDO VISUALIZAÇÕES DOS RESULTADOS SISTEMÁTICOS")
    print("="*60)
    
    if df_results is None or len(df_results) == 0:
        print("❌ Nenhum dado disponível para visualização")
        return
    
    # Configurar estilo
    plt.style.use('default')
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('Análise Sistemática: KD Techniques × Prompt Strategies', fontsize=16, fontweight='bold')
    
    # 1. Heatmap F1 Scores
    if pivot_table is not None and not pivot_table.empty:
        sns.heatmap(pivot_table, annot=True, fmt='.3f', cmap='YlOrRd', ax=axes[0,0])
        axes[0,0].set_title('F1-Scores por Combinação')
        axes[0,0].set_xlabel('Estratégia de Prompt')
        axes[0,0].set_ylabel('Técnica de KD')
    
    # 2. Comparação por técnica KD
    kd_means = df_results.groupby('kd_technique')[['f1_score', 'exact_match']].mean()
    kd_means.plot(kind='bar', ax=axes[0,1], color=['skyblue', 'lightcoral'])
    axes[0,1].set_title('Performance Média por Técnica KD')
    axes[0,1].set_xlabel('Técnica de Knowledge Distillation')
    axes[0,1].set_ylabel('Score')
    axes[0,1].legend(['F1-Score', 'Exact Match'])
    axes[0,1].tick_params(axis='x', rotation=45)
    
    # 3. Comparação por estratégia de prompt
    prompt_means = df_results.groupby('prompt_strategy')[['f1_score', 'exact_match']].mean()
    prompt_means.plot(kind='bar', ax=axes[1,0], color=['lightgreen', 'orange'])
    axes[1,0].set_title('Performance Média por Estratégia de Prompt')
    axes[1,0].set_xlabel('Estratégia de Prompt')
    axes[1,0].set_ylabel('Score')
    axes[1,0].legend(['F1-Score', 'Exact Match'])
    axes[1,0].tick_params(axis='x', rotation=45)
    
    # 4. Scatter plot F1 vs Exact Match
    colors = plt.cm.Set3(np.linspace(0, 1, len(df_results)))
    scatter = axes[1,1].scatter(df_results['f1_score'], df_results['exact_match'], 
                              c=colors, s=100, alpha=0.7)
    axes[1,1].set_xlabel('F1-Score')
    axes[1,1].set_ylabel('Exact Match')
    axes[1,1].set_title('F1-Score vs Exact Match')
    
    # Anotar pontos no scatter plot
    for i, row in df_results.iterrows():
        axes[1,1].annotate(f"{row['kd_technique'][:8]}\n{row['prompt_strategy'][:8]}", 
                          (row['f1_score'], row['exact_match']),
                          xytext=(5, 5), textcoords='offset points', fontsize=8)
    
    plt.tight_layout()
    
    # Salvar visualização
    viz_file = f"./systematic_results/systematic_visualization_{datetime.now().strftime('%Y%m%d_%H%M%S')}.png"
    plt.savefig(viz_file, dpi=300, bbox_inches='tight')
    print(f"📊 Visualização salva em: {viz_file}")
    
    plt.show()
    
    # Relatório textual
    print(f"\n📋 RELATÓRIO FINAL:")
    print(f"   Total de experimentos: {len(df_results)}")
    print(f"   Melhor F1-Score: {df_results['f1_score'].max():.3f}")
    print(f"   Melhor Exact Match: {df_results['exact_match'].max():.3f}")
    print(f"   Técnica KD mais eficaz: {df_results.groupby('kd_technique')['f1_score'].mean().idxmax()}")
    print(f"   Estratégia Prompt mais eficaz: {df_results.groupby('prompt_strategy')['f1_score'].mean().idxmax()}")

    # Execute visualization
if 'df_systematic' in locals() and 'pivot_systematic' in locals():
    visualize_systematic_results(df_systematic, pivot_systematic)

print(f"\n🎉 EXPERIMENTOS SISTEMÁTICOS CONCLUÍDOS!")
print(f"Execute as células individuais para rodar experimentos específicos.")

## Técnicas avançadas (extensões futuras)
Classe com métodos para avaliação GLUE, RAG e outras extensões científicas.

In [22]:
class AdvancedTechniques:
    """Técnicas avançadas para extensões futuras do experimento científico"""

    @staticmethod
    def implement_glue_benchmark(config, logger, models_dict):
        """
        Implementação de avaliação GLUE multi-tarefa

        Args:
            config: Configuração do experimento
            logger: Logger científico
            models_dict: Dicionário com modelos para avaliar

        Returns:
            dict: Resultados GLUE por tarefa e modelo
        """
        print("🎯 Implementando GLUE Benchmark Multi-tarefa...")

        # Tarefas GLUE implementáveis com recursos limitados
        glue_tasks = {
            'cola': {
                'dataset': 'glue',
                'config_name': 'cola',
                'metric': 'matthews_correlation',
                'text_field': 'sentence',
                'label_field': 'label',
                'task_type': 'classification'
            },
            'sst2': {
                'dataset': 'glue',
                'config_name': 'sst2',
                'metric': 'accuracy',
                'text_field': 'sentence',
                'label_field': 'label',
                'task_type': 'classification'
            },
            'mrpc': {
                'dataset': 'glue',
                'config_name': 'mrpc',
                'metric': 'f1',
                'text_field': ['sentence1', 'sentence2'],
                'label_field': 'label',
                'task_type': 'pair_classification'
            },
            'rte': {
                'dataset': 'glue',
                'config_name': 'rte',
                'metric': 'accuracy',
                'text_field': ['sentence1', 'sentence2'],
                'label_field': 'label',
                'task_type': 'pair_classification'
            }
        }

        glue_results = {}

        for task_name, task_config in glue_tasks.items():
            print(f"  📋 Avaliando tarefa: {task_name.upper()}")

            try:
                # Carregar dataset da tarefa
                dataset = load_dataset(task_config['dataset'], task_config['config_name'])
                eval_data = dataset['validation'].shuffle(seed=42).select(range(min(200, len(dataset['validation']))))

                # Carregar métrica
                metric = load(task_config['metric'])

                task_results = {}

                for model_name, (model, tokenizer) in models_dict.items():
                    print(f"    🔬 Modelo: {model_name}")

                    predictions = []
                    references = []

                    for example in tqdm(eval_data, desc=f"{model_name}-{task_name}"):
                        # Criar prompt específico para a tarefa
                        prompt = AdvancedTechniques._create_glue_prompt(example, task_name, task_config)

                        # Tokenizar e gerar
                        inputs = tokenizer(prompt, return_tensors="pt", max_length=config.MAX_LENGTH,
                                         truncation=True).to(config.DEVICE)

                        with torch.no_grad():
                            outputs = model.generate(
                                **inputs,
                                max_new_tokens=10,
                                temperature=0.1,
                                do_sample=False,
                                pad_token_id=tokenizer.eos_token_id
                            )

                        # Extrair predição
                        response = tokenizer.decode(outputs[0], skip_special_tokens=True)
                        predicted_label = AdvancedTechniques._extract_glue_prediction(
                            response[len(prompt):].strip(), task_name
                        )

                        predictions.append(predicted_label)
                        references.append(example[task_config['label_field']])

                    # Calcular métrica
                    if task_config['metric'] == 'matthews_correlation':
                        score = metric.compute(predictions=predictions, references=references)['matthews_correlation']
                    elif task_config['metric'] == 'accuracy':
                        score = metric.compute(predictions=predictions, references=references)['accuracy']
                    elif task_config['metric'] == 'f1':
                        score = metric.compute(predictions=predictions, references=references)['f1']
                    else:
                        score = 0.0

                    task_results[model_name] = {
                        'score': score,
                        'predictions_sample': predictions[:5],
                        'references_sample': references[:5]
                    }

                    print(f"      ✅ {task_config['metric']}: {score:.3f}")

                glue_results[task_name] = task_results

            except Exception as e:
                print(f"    ❌ Erro na tarefa {task_name}: {e}")
                glue_results[task_name] = {'error': str(e)}

        # Calcular GLUE score agregado
        glue_scores = {}
        for model_name in models_dict.keys():
            valid_scores = []
            for task_name, task_results in glue_results.items():
                if model_name in task_results and 'score' in task_results[model_name]:
                    valid_scores.append(task_results[model_name]['score'])

            glue_scores[model_name] = np.mean(valid_scores) if valid_scores else 0.0

        logger.log_phase("GLUE_BENCHMARK", {
            'status': 'Concluído',
            'tasks_evaluated': list(glue_tasks.keys()),
            'glue_scores': glue_scores,
            'detailed_results': glue_results
        })

        return {'glue_scores': glue_scores, 'detailed_results': glue_results}

    @staticmethod
    def _create_glue_prompt(example, task_name, task_config):
        """Cria prompt específico para tarefa GLUE"""
        if task_name == 'cola':
            return f"A seguinte frase é gramaticalmente aceitável?\nFrase: {example['sentence']}\nResposta (aceitável/inaceitável):"
        elif task_name == 'sst2':
            return f"Qual é o sentimento desta frase?\nFrase: {example['sentence']}\nResposta (positivo/negativo):"
        elif task_name == 'mrpc':
            return f"As duas frases a seguir têm o mesmo significado?\nFrase 1: {example['sentence1']}\nFrase 2: {example['sentence2']}\nResposta (sim/não):"
        elif task_name == 'rte':
            return f"A hipótese pode ser inferida da premissa?\nPremissa: {example['sentence1']}\nHipótese: {example['sentence2']}\nResposta (sim/não):"
        else:
            return str(example)

    @staticmethod
    def _extract_glue_prediction(response, task_name):
        """Extrai predição do texto gerado para tarefas GLUE"""
        response_lower = response.lower().strip()

        if task_name == 'cola':
            if 'aceitável' in response_lower and 'inaceitável' not in response_lower:
                return 1
            elif 'inaceitável' in response_lower:
                return 0
            else:
                return 1 if 'sim' in response_lower or 'yes' in response_lower else 0

        elif task_name == 'sst2':
            if 'positivo' in response_lower:
                return 1
            elif 'negativo' in response_lower:
                return 0
            else:
                return 1 if any(word in response_lower for word in ['bom', 'ótimo', 'excelente']) else 0

        elif task_name in ['mrpc', 'rte']:
            if 'sim' in response_lower or 'yes' in response_lower:
                return 1
            elif 'não' in response_lower or 'no' in response_lower:
                return 0
            else:
                return 1 if any(word in response_lower for word in ['verdade', 'correto', 'certo']) else 0

        return 0

    @staticmethod
    def implement_retrieval_augmented_generation(config, logger, model, tokenizer, eval_dataset, knowledge_base=None):
        """
        Implementação de RAG para ICL dinâmico

        Args:
            config: Configuração do experimento
            logger: Logger científico
            model: Modelo para avaliação
            tokenizer: Tokenizer do modelo
            eval_dataset: Dataset de avaliação
            knowledge_base: Base de conhecimento externa (opcional)

        Returns:
            dict: Resultados da avaliação RAG
        """
        print("🔍 Implementando Retrieval-Augmented Generation...")

        # Criar knowledge base se não fornecida
        if knowledge_base is None:
            print("  📚 Criando knowledge base a partir do dataset SQuAD...")
            squad_train = load_dataset("squad", split="train")

            # Criar índice simples baseado em similaridade TF-IDF
            from sklearn.feature_extraction.text import TfidfVectorizer
            from sklearn.metrics.pairwise import cosine_similarity

            # Preparar corpus
            contexts = [example['context'] for example in squad_train.select(range(5000))]
            questions = [example['question'] for example in squad_train.select(range(5000))]
            answers = [example['answers']['text'][0] for example in squad_train.select(range(5000))]

            # Vetorizar contextos
            vectorizer = TfidfVectorizer(max_features=1000, stop_words='english')
            context_vectors = vectorizer.fit_transform(contexts)

            knowledge_base = {
                'contexts': contexts,
                'questions': questions,
                'answers': answers,
                'vectorizer': vectorizer,
                'vectors': context_vectors
            }

        print("  🎯 Avaliando com RAG...")
        predictions = []
        references = []
        retrieval_scores = []

        squad_metric = load("squad")

        for example in tqdm(eval_dataset.select(range(100)), desc="RAG Evaluation"):
            # 1. Recuperação de contextos relevantes
            query_vector = knowledge_base['vectorizer'].transform([example['question']])
            similarities = cosine_similarity(query_vector, knowledge_base['vectors']).flatten()

            # Top-k contextos mais similares
            top_k = 3
            top_indices = similarities.argsort()[-top_k:][::-1]

            # 2. Construir prompt com contextos recuperados
            retrieved_contexts = []
            for idx in top_indices:
                retrieved_contexts.append({
                    'context': knowledge_base['contexts'][idx],
                    'question': knowledge_base['questions'][idx],
                    'answer': knowledge_base['answers'][idx],
                    'similarity': similarities[idx]
                })

            # Construir prompt RAG
            rag_prompt = AdvancedTechniques._create_rag_prompt(example, retrieved_contexts)

            # 3. Geração
            inputs = tokenizer(rag_prompt, return_tensors="pt", max_length=config.MAX_LENGTH*2,
                             truncation=True).to(config.DEVICE)

            with torch.no_grad():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=60,
                    temperature=0.7,
                    do_sample=True,
                    pad_token_id=tokenizer.eos_token_id
                )

            # Extrair resposta
            response = tokenizer.decode(outputs[0], skip_special_tokens=True)
            predicted_answer = response[len(rag_prompt):].strip()

            predictions.append({
                'id': example['id'],
                'prediction_text': predicted_answer
            })
            references.append({
                'id': example['id'],
                'answers': example['answers']
            })

            # Score de recuperação (média das similaridades dos contextos usados)
            retrieval_scores.append(np.mean([ctx['similarity'] for ctx in retrieved_contexts]))

        # Calcular métricas
        rag_results = squad_metric.compute(predictions=predictions, references=references)
        rag_results.update({
            'avg_retrieval_score': np.mean(retrieval_scores),
            'std_retrieval_score': np.std(retrieval_scores),
            'num_examples': len(predictions)
        })

        logger.log_phase("RAG_EVALUATION", {
            'status': 'Concluído',
            'results': rag_results,
            'knowledge_base_size': len(knowledge_base['contexts'])
        })

        print(f"  ✅ RAG F1: {rag_results['f1']:.2f}, EM: {rag_results['exact_match']:.2f}")
        print(f"  Retrieval Score: {rag_results['avg_retrieval_score']:.3f}")

        return rag_results

    @staticmethod
    def _create_rag_prompt(example, retrieved_contexts):
        """Cria prompt RAG com contextos recuperados"""
        prompt = "Baseando-se nos seguintes contextos relevantes, responda à pergunta:\n\n"

        # Adicionar contextos recuperados
        for i, ctx in enumerate(retrieved_contexts, 1):
            prompt += f"Contexto {i}:\n{ctx['context']}\n"
            prompt += f"Exemplo - P: {ctx['question']}\nR: {ctx['answer']}\n\n"

        # Adicionar pergunta atual
        prompt += f"Agora responda:\nContexto: {example['context']}\n"
        prompt += f"Pergunta: {example['question']}\nResposta:"

        return prompt

    @staticmethod
    def implement_cola_rte_evaluation(config, logger, models_dict):
        """
        Implementação específica de CoLA e RTE para análise linguística

        Args:
            config: Configuração do experimento
            logger: Logger científico
            models_dict: Dicionário com modelos

        Returns:
            dict: Análises linguísticas detalhadas
        """
        print("🧠 Implementando Avaliação Linguística (CoLA + RTE)...")

        linguistic_results = {}

        # === COLA: Corpus of Linguistic Acceptability ===
        print("  📝 Avaliando CoLA (Aceitabilidade Linguística)...")
        try:
            cola_dataset = load_dataset("glue", "cola")
            cola_eval = cola_dataset['validation'].shuffle(seed=42).select(range(200))
            cola_metric = load("matthews_correlation")

            cola_results = {}
            linguistic_patterns = defaultdict(list)

            for model_name, (model, tokenizer) in models_dict.items():
                print(f"    🔬 Modelo: {model_name}")

                predictions = []
                references = []
                confidence_scores = []

                for example in tqdm(cola_eval, desc=f"{model_name}-CoLA"):
                    sentence = example['sentence']

                    # Prompt para aceitabilidade linguística
                    prompt = f"""Analise se a seguinte frase é gramaticalmente aceitável em português:
Frase: "{sentence}"

Considere:
- Estrutura sintática
- Concordância verbal/nominal
- Ordem das palavras
- Uso adequado de preposições

Resposta (aceitável/inaceitável):"""

                    inputs = tokenizer(prompt, return_tensors="pt", max_length=config.MAX_LENGTH,
                                     truncation=True).to(config.DEVICE)

                    # Obter probabilidades para análise de confiança
                    with torch.no_grad():
                        outputs = model(**inputs)
                        logits = outputs.logits[0, -1, :]  # Último token

                        # Tokens para "aceitável" vs "inaceitável"
                        acceptable_tokens = tokenizer.encode("aceitável", add_special_tokens=False)
                        unacceptable_tokens = tokenizer.encode("inaceitável", add_special_tokens=False)

                        if acceptable_tokens and unacceptable_tokens:
                            acc_prob = torch.softmax(logits, dim=-1)[acceptable_tokens[0]]
                            unacc_prob = torch.softmax(logits, dim=-1)[unacceptable_tokens[0]]
                            confidence = max(acc_prob, unacc_prob).item()
                            prediction = 1 if acc_prob > unacc_prob else 0
                        else:
                            # Fallback para geração
                            gen_outputs = model.generate(**inputs, max_new_tokens=5, temperature=0.1)
                            response = tokenizer.decode(gen_outputs[0], skip_special_tokens=True)
                            prediction = 1 if 'aceitável' in response.lower() else 0
                            confidence = 0.5

                    predictions.append(prediction)
                    references.append(example['label'])
                    confidence_scores.append(confidence)

                    # Analisar padrões linguísticos
                    if len(sentence.split()) < 5:
                        linguistic_patterns['short_sentences'].append((sentence, prediction, example['label']))
                    elif any(word in sentence.lower() for word in ['que', 'quem', 'onde', 'quando']):
                        linguistic_patterns['interrogatives'].append((sentence, prediction, example['label']))
                    elif any(word in sentence.lower() for word in ['não', 'nunca', 'nenhum']):
                        linguistic_patterns['negation'].append((sentence, prediction, example['label']))

                # Calcular Matthews Correlation
                mcc_score = cola_metric.compute(predictions=predictions, references=references)['matthews_correlation']

                cola_results[model_name] = {
                    'matthews_correlation': mcc_score,
                    'accuracy': np.mean([p == r for p, r in zip(predictions, references)]),
                    'avg_confidence': np.mean(confidence_scores),
                    'linguistic_patterns': dict(linguistic_patterns)
                }

                print(f"      ✅ MCC: {mcc_score:.3f}, Acc: {cola_results[model_name]['accuracy']:.3f}")

            linguistic_results['cola'] = cola_results

        except Exception as e:
            print(f"    ❌ Erro CoLA: {e}")
            linguistic_results['cola'] = {'error': str(e)}

        # === RTE: Recognizing Textual Entailment ===
        print("  🔗 Avaliando RTE (Inferência Textual)...")
        try:
            rte_dataset = load_dataset("glue", "rte")
            rte_eval = rte_dataset['validation'].shuffle(seed=42).select(range(200))
            rte_metric = load("accuracy")

            rte_results = {}
            entailment_patterns = defaultdict(list)

            for model_name, (model, tokenizer) in models_dict.items():
                print(f"    🔬 Modelo: {model_name}")

                predictions = []
                references = []
                reasoning_quality = []

                for example in tqdm(rte_eval, desc=f"{model_name}-RTE"):
                    premise = example['sentence1']
                    hypothesis = example['sentence2']

                    # Prompt para inferência textual
                    prompt = f"""Analise se a hipótese pode ser logicamente inferida da premissa:

Premissa: "{premise}"
Hipótese: "{hypothesis}"

Pense passo a passo:
1. Quais informações a premissa fornece?
2. O que a hipótese afirma?
3. A hipótese é uma consequência lógica da premissa?

Resposta (sim/não):"""

                    inputs = tokenizer(prompt, return_tensors="pt", max_length=config.MAX_LENGTH*2,
                                     truncation=True).to(config.DEVICE)

                    with torch.no_grad():
                        outputs = model.generate(
                            **inputs,
                            max_new_tokens=100,
                            temperature=0.3,
                            do_sample=True,
                            pad_token_id=tokenizer.eos_token_id
                        )

                    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
                    reasoning = response[len(prompt):].strip()

                    # Extrair predição
                    prediction = 1 if any(word in reasoning.lower() for word in ['sim', 'yes', 'verdade']) else 0
                    predictions.append(prediction)
                    references.append(example['label'])

                    # Avaliar qualidade do raciocínio
                    reasoning_score = AdvancedTechniques._evaluate_reasoning_quality(reasoning)
                    reasoning_quality.append(reasoning_score)

                    # Analisar padrões de inferência
                    if 'não' in premise.lower() or 'não' in hypothesis.lower():
                        entailment_patterns['negation'].append((premise, hypothesis, prediction, example['label']))
                    elif any(word in hypothesis.lower() for word in ['todos', 'alguns', 'muitos']):
                        entailment_patterns['quantifiers'].append((premise, hypothesis, prediction, example['label']))
                    elif len(hypothesis.split()) > 2 * len(premise.split()):
                        entailment_patterns['hypothesis_expansion'].append((premise, hypothesis, prediction, example['label']))

                # Calcular métricas
                accuracy = rte_metric.compute(predictions=predictions, references=references)['accuracy']

                rte_results[model_name] = {
                    'accuracy': accuracy,
                    'avg_reasoning_quality': np.mean(reasoning_quality),
                    'entailment_patterns': dict(entailment_patterns)
                }

                print(f"      ✅ Acc: {accuracy:.3f}, Reasoning: {rte_results[model_name]['avg_reasoning_quality']:.3f}")

            linguistic_results['rte'] = rte_results

        except Exception as e:
            print(f"    ❌ Erro RTE: {e}")
            linguistic_results['rte'] = {'error': str(e)}

        logger.log_phase("LINGUISTIC_ANALYSIS", {
            'status': 'Concluído',
            'cola_results': linguistic_results.get('cola', {}),
            'rte_results': linguistic_results.get('rte', {})
        })

        return linguistic_results

    @staticmethod
    def _evaluate_reasoning_quality(reasoning_text):
        """Avalia qualidade do raciocínio textual"""
        score = 0.0
        reasoning_lower = reasoning_text.lower()

        # Presença de estrutura lógica
        if any(marker in reasoning_lower for marker in ['porque', 'portanto', 'assim', 'logo']):
            score += 0.3

        # Referência às premissas
        if any(word in reasoning_lower for word in ['premissa', 'afirma', 'diz']):
            score += 0.2

        # Referência à hipótese
        if any(word in reasoning_lower for word in ['hipótese', 'conclusão', 'implica']):
            score += 0.2

        # Estrutura step-by-step
        if any(pattern in reasoning_lower for pattern in ['1.', '2.', '3.', 'primeiro', 'segundo']):
            score += 0.3

        return min(score, 1.0)

    @staticmethod
    def implement_progressive_distillation(config, logger, teacher_model, teacher_tokenizer, kd_dataset):
        """
        Implementação de destilação progressiva multi-estágio

        Args:
            config: Configuração do experimento
            logger: Logger científico
            teacher_model: Modelo professor inicial
            teacher_tokenizer: Tokenizer do professor
            kd_dataset: Dataset para destilação

        Returns:
            list: Lista de modelos intermediários e final
        """
        print("🔄 Implementando Destilação Progressiva Multi-estágio...")

        # Configuração dos estágios
        stages_config = [
            {'name': 'Estágio 1', 'temperature': 5.0, 'alpha': 0.8, 'epochs': 1, 'lr': 5e-4},
            {'name': 'Estágio 2', 'temperature': 3.0, 'alpha': 0.7, 'epochs': 1, 'lr': 3e-4},
            {'name': 'Estágio 3', 'temperature': 2.0, 'alpha': 0.6, 'epochs': 2, 'lr': 2e-4},
            {'name': 'Refinamento', 'temperature': 1.5, 'alpha': 0.5, 'epochs': 1, 'lr': 1e-4}
        ]

        progressive_models = []
        current_teacher = teacher_model
        current_teacher_tokenizer = teacher_tokenizer

        # Configurar LoRA comum para todos os estágios
        from peft import LoraConfig, get_peft_model, TaskType

        base_lora_config = LoraConfig(
            r=config.LORA_R,
            lora_alpha=config.LORA_ALPHA,
            target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
            lora_dropout=config.LORA_DROPOUT,
            bias="none",
            task_type=TaskType.CAUSAL_LM
        )

        for stage_idx, stage_config in enumerate(stages_config):
            print(f"  🎯 {stage_config['name']} - T={stage_config['temperature']}, α={stage_config['alpha']}")

            # Carregar modelo estudante para este estágio
            stage_student = AutoModelForCausalLM.from_pretrained(
                config.STUDENT_MODEL_ID,
                device_map="auto",
                torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32
            )

            # Aplicar LoRA
            stage_student_peft = get_peft_model(stage_student, base_lora_config)
            stage_student_peft.print_trainable_parameters()

            # Setup otimizador para este estágio
            optimizer = AdamW(stage_student_peft.parameters(), lr=stage_config['lr'])

            # Métricas do estágio
            stage_metrics = {
                'stage_name': stage_config['name'],
                'epochs': [],
                'kd_losses': [],
                'ce_losses': [],
                'total_losses': [],
                'temperature': stage_config['temperature'],
                'alpha': stage_config['alpha']
            }

            # Treinar neste estágio
            current_teacher.eval()
            stage_student_peft.train()

            # Dataset progressivamente mais desafiador
            stage_data_size = min(500 + stage_idx * 200, len(kd_dataset))
            stage_dataset = kd_dataset.shuffle(seed=42 + stage_idx).select(range(stage_data_size))

            for epoch in range(stage_config['epochs']):
                epoch_kd_losses = []
                epoch_ce_losses = []
                epoch_total_losses = []

                # Amostra do dataset para esta época
                epoch_data = stage_dataset.shuffle(seed=42 + epoch).select(
                    range(min(300, len(stage_dataset)))
                )

                for step, batch in enumerate(tqdm(epoch_data, desc=f"{stage_config['name']} - Época {epoch+1}")):
                    # Criar prompt
                    prompt = f"Contexto: {batch['context']}\n\nPergunta: {batch['question']}\n\nResposta: {batch['answers']['text'][0]}"

                    # Tokenização
                    teacher_inputs = current_teacher_tokenizer(
                        prompt, return_tensors="pt", max_length=config.MAX_LENGTH,
                        truncation=True, padding=True
                    ).to(config.DEVICE)

                    student_inputs = teacher_tokenizer(  # Usar mesmo tokenizer
                        prompt, return_tensors="pt", max_length=config.MAX_LENGTH,
                        truncation=True, padding=True
                    ).to(config.DEVICE)

                    # Forward pass
                    with torch.no_grad():
                        teacher_outputs = current_teacher(**teacher_inputs)

                    student_outputs = stage_student_peft(**student_inputs, labels=student_inputs['input_ids'])

                    # KD Loss com configuração do estágio
                    if student_outputs.logits.shape[-1] == teacher_outputs.logits.shape[-1]:
                        # Soft targets do professor
                        teacher_probs = F.softmax(teacher_outputs.logits / stage_config['temperature'], dim=-1)
                        student_log_probs = F.log_softmax(student_outputs.logits / stage_config['temperature'], dim=-1)

                        # KL Divergence
                        kd_loss = F.kl_div(student_log_probs, teacher_probs, reduction='batchmean')
                        kd_loss *= (stage_config['temperature'] ** 2)

                        # Combinar com CE
                        ce_loss = student_outputs.loss
                        total_loss = stage_config['alpha'] * kd_loss + (1 - stage_config['alpha']) * ce_loss

                        kd_loss_val = kd_loss.item()
                        ce_loss_val = ce_loss.item()
                    else:
                        # Fallback para CE apenas
                        total_loss = student_outputs.loss
                        kd_loss_val = 0.0
                        ce_loss_val = total_loss.item()

                    # Backward pass
                    total_loss.backward()
                    optimizer.step()
                    optimizer.zero_grad()

                    # Coletar métricas
                    epoch_kd_losses.append(kd_loss_val)
                    epoch_ce_losses.append(ce_loss_val)
                    epoch_total_losses.append(total_loss.item())

                    if step % 50 == 0:
                        print(f"      Step {step}: KD={kd_loss_val:.4f}, CE={ce_loss_val:.4f}")

                # Salvar métricas da época
                stage_metrics['epochs'].append(epoch + 1)
                stage_metrics['kd_losses'].append(np.mean(epoch_kd_losses))
                stage_metrics['ce_losses'].append(np.mean(epoch_ce_losses))
                stage_metrics['total_losses'].append(np.mean(epoch_total_losses))

            # Salvar modelo do estágio
            stage_output_path = f"{config.MODELS_DIR}/progressive_stage_{stage_idx+1}_{stage_config['name'].lower().replace(' ', '_')}"
            stage_student_peft.save_pretrained(stage_output_path)

            # Preparar para próximo estágio (estudante vira professor)
            if stage_idx < len(stages_config) - 1:  # Se não for último estágio
                current_teacher = stage_student_peft
                current_teacher_tokenizer = teacher_tokenizer
                print(f"    🔄 Estudante do {stage_config['name']} vira professor do próximo estágio")

            # Armazenar modelo e métricas
            progressive_models.append({
                'stage': stage_idx + 1,
                'model': stage_student_peft,
                'tokenizer': teacher_tokenizer,
                'metrics': stage_metrics,
                'config': stage_config,
                'output_path': stage_output_path
            })

            print(f"    ✅ {stage_config['name']} concluído - Loss final: {stage_metrics['total_losses'][-1]:.4f}")

        # Log do processo completo
        logger.log_phase("PROGRESSIVE_DISTILLATION", {
            'status': 'Concluído',
            'num_stages': len(stages_config),
            'stages_config': stages_config,
            'final_models': len(progressive_models),
            'improvement_per_stage': [
                model['metrics']['total_losses'][-1] for model in progressive_models
            ]
        })

        print(f"  ✅ Destilação Progressiva concluída - {len(progressive_models)} modelos intermediários criados")

        return progressive_models

# Exemplo de uso:
if __name__ == "__main__":
    # Para execução rápida apenas com avaliação:
    # results = run_scientific_experiment(training=False, evaluation=True)

    # Para execução completa com treinamento:
    # results = run_scientific_experiment(training=True, evaluation=True)

    # Para testar técnicas avançadas:
    # runner = ScientificExperimentRunner()
    # config = runner.config
    # logger = runner.logger
    # datasets, _ = runner.dataset_manager.load_and_analyze_data()
    # models_dict = runner._prepare_evaluation_models({})

    # Exemplo de uso das técnicas avançadas:
    # glue_results = AdvancedTechniques.implement_glue_benchmark(config, logger, models_dict)
    # rag_results = AdvancedTechniques.implement_retrieval_augmented_generation(
    #     config, logger, models_dict['SLM Base'][0], models_dict['SLM Base'][1], datasets['evaluation']
    # )
    # linguistic_results = AdvancedTechniques.implement_cola_rte_evaluation(config, logger, models_dict)

    print("📋 Para executar, descomente uma das linhas acima ou use:")
    print("results = run_scientific_experiment(training=False, evaluation=True)")

def run_advanced_techniques_demo():
    """
    Função demonstrativa para executar as técnicas avançadas implementadas
    """
    print("🚀 Executando Demonstração das Técnicas Avançadas...")

    # Setup básico
    runner = ScientificExperimentRunner()
    config = runner.config
    logger = runner.logger

    # Carregar dados
    datasets, dataset_stats = runner.dataset_manager.load_and_analyze_data()

    # Preparar modelos para teste
    models_dict = runner._prepare_evaluation_models({})

    if not models_dict:
        print("❌ Nenhum modelo disponível para demonstração")
        return

    results = {}

    print("\n" + "="*60)
    print("1️⃣ GLUE BENCHMARK")
    print("="*60)
    try:
        glue_results = AdvancedTechniques.implement_glue_benchmark(config, logger, models_dict)
        results['glue'] = glue_results

        # Mostrar resultados resumidos
        print("\nResumo GLUE:")
        for model_name, score in glue_results['glue_scores'].items():
            print(f"  {model_name}: {score:.3f}")

    except Exception as e:
        print(f"❌ Erro GLUE: {e}")
        results['glue'] = {'error': str(e)}

    print("\n" + "="*60)
    print("2️⃣ RETRIEVAL-AUGMENTED GENERATION")
    print("="*60)
    try:
        # Usar primeiro modelo disponível
        first_model_name = list(models_dict.keys())[0]
        model, tokenizer = models_dict[first_model_name]

        rag_results = AdvancedTechniques.implement_retrieval_augmented_generation(
            config, logger, model, tokenizer, datasets['evaluation']
        )
        results['rag'] = rag_results

        print(f"\nResultados RAG:")
        print(f"  F1-Score: {rag_results['f1']:.3f}")
        print(f"  Exact Match: {rag_results['exact_match']:.3f}")
        print(f"  Retrieval Score: {rag_results['avg_retrieval_score']:.3f}")

    except Exception as e:
        print(f"❌ Erro RAG: {e}")
        results['rag'] = {'error': str(e)}

    print("\n" + "="*60)
    print("3️⃣ ANÁLISE LINGUÍSTICA (CoLA + RTE)")
    print("="*60)
    try:
        linguistic_results = AdvancedTechniques.implement_cola_rte_evaluation(config, logger, models_dict)
        results['linguistic'] = linguistic_results

        print(f"\nResultados Linguísticos:")
        if 'cola' in linguistic_results:
            print("  CoLA (Aceitabilidade Linguística):")
            for model_name, result in linguistic_results['cola'].items():
                if 'matthews_correlation' in result:
                    print(f"    {model_name}: MCC = {result['matthews_correlation']:.3f}")

        if 'rte' in linguistic_results:
            print("  RTE (Inferência Textual):")
            for model_name, result in linguistic_results['rte'].items():
                if 'accuracy' in result:
                    print(f"    {model_name}: Acc = {result['accuracy']:.3f}")

    except Exception as e:
        print(f"❌ Erro Análise Linguística: {e}")
        results['linguistic'] = {'error': str(e)}

    print("\n" + "="*60)
    print("4️⃣ DESTILAÇÃO PROGRESSIVA")
    print("="*60)
    print("⚠️ Destilação Progressiva requer recursos computacionais significativos")
    print("💡 Para testar, use: progressive_models = AdvancedTechniques.implement_progressive_distillation(...)")

    # Salvar resultados da demonstração
    demo_report_path = f"{config.REPORTS_DIR}/advanced_techniques_demo.json"
    with open(demo_report_path, 'w') as f:
        json.dump(results, f, indent=2, default=str)

    print(f"\n✅ Demonstração concluída!")
    print(f"📁 Resultados salvos em: {demo_report_path}")

    return results

📋 Para executar, descomente uma das linhas acima ou use:
results = run_scientific_experiment(training=False, evaluation=True)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import json
from pathlib import Path

# Configurações gerais de plot
plt.style.use('default')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

class SLMResultsVisualizer:
    def __init__(self, base_dir="scientific_results"):
        self.base_dir = Path(base_dir)
        self.reports_dir = self.base_dir / "reports"
        self.figures_dir = self.base_dir / "figures"

        # Criar pasta de figuras se não existir
        self.figures_dir.mkdir(exist_ok=True)

        self.load_data()

    def load_data(self):
        """Carrega todos os dados dos arquivos da pasta reports"""
        # Carregar dados dos CSVs
        self.f1_scores = pd.read_csv(self.reports_dir / "f1_results.csv", index_col=0)
        self.exact_match = pd.read_csv(self.reports_dir / "exact_match_results.csv", index_col=0)

        # Carregar dados do JSON
        with open(self.reports_dir / "experiment_log.json", 'r', encoding='utf-8') as f:
            self.experiment_data = json.load(f)

        # Extrair estatísticas do dataset
        self.dataset_stats = self.experiment_data['phases']['DATASET_ANALYSIS']['data']['stats']
        self.results_summary = self.experiment_data['phases']['COMPREHENSIVE_EVALUATION']['data']['results_summary']

    def plot_f1_distribution(self):
        """Plota a distribuição dos F1-Scores por estratégia"""
        fig, ax = plt.subplots(figsize=(14, 8))

        # Preparar dados para boxplot
        strategies = self.f1_scores.index
        f1_values = self.f1_scores['SLM Base'].values

        # Criar boxplot (mesmo com um valor por estratégia)
        box_data = [[val] for val in f1_values]
        bp = ax.boxplot(box_data, labels=strategies, patch_artist=True,
                       boxprops=dict(facecolor='lightcoral', alpha=0.7),
                       medianprops=dict(color='darkred', linewidth=2))

        ax.set_title('Distribuição de F1-Scores por Estratégia', fontsize=16, pad=20)
        ax.set_ylabel('F1-Score', fontsize=12)
        ax.set_xlabel('Estratégias de Prompt Engineering', fontsize=12)
        ax.grid(True, alpha=0.3)

        # Rotacionar labels do eixo x
        plt.setp(ax.get_xticklabels(), rotation=45, ha='right')

        # Adicionar valores nos pontos
        for i, val in enumerate(f1_values):
            ax.text(i+1, val + 0.5, f'{val:.2f}', ha='center', va='bottom', fontweight='bold')

        plt.tight_layout()
        return fig

    def plot_performance_comparison(self):
        """Plota comparação de performance entre modelos e estratégias"""
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))

        # Performance média por modelo
        model_performance = self.f1_scores.mean()
        ax1.bar(model_performance.index, model_performance.values,
                color='skyblue', alpha=0.8, edgecolor='navy')
        ax1.set_title('Performance Média por Modelo (F1-Score)', fontsize=14)
        ax1.set_ylabel('F1-Score', fontsize=12)
        ax1.grid(True, alpha=0.3)

        # Adicionar valores nas barras
        for i, val in enumerate(model_performance.values):
            ax1.text(i, val + 0.1, f'{val:.2f}', ha='center', va='bottom', fontweight='bold')

        # Performance média por estratégia
        strategy_performance = self.f1_scores.mean(axis=1)
        colors = plt.cm.Set2(np.linspace(0, 1, len(strategy_performance)))
        bars = ax2.bar(range(len(strategy_performance)), strategy_performance.values,
                      color=colors, alpha=0.8, edgecolor='black')
        ax2.set_title('Performance Média por Estratégia (F1-Score)', fontsize=14)
        ax2.set_ylabel('F1-Score', fontsize=12)
        ax2.set_xticks(range(len(strategy_performance)))
        ax2.set_xticklabels(strategy_performance.index, rotation=45, ha='right')
        ax2.grid(True, alpha=0.3)

        # Adicionar valores nas barras
        for i, val in enumerate(strategy_performance.values):
            ax2.text(i, val + 0.2, f'{val:.2f}', ha='center', va='bottom', fontweight='bold')

        plt.tight_layout()
        return fig

    def plot_heatmap_performance(self):
        """Plota heatmap da matriz de performance"""
        fig, ax = plt.subplots(figsize=(10, 8))

        # Criar heatmap
        sns.heatmap(self.f1_scores, annot=True, fmt='.2f', cmap='YlOrRd',
                   ax=ax, cbar_kws={'label': 'F1-Score'},
                   linewidths=1, linecolor='white')

        ax.set_title('Matriz de Performance (F1-Score)\nModelos vs Estratégias de Prompt',
                     fontsize=14, pad=20)
        ax.set_xlabel('Modelos', fontsize=12)
        ax.set_ylabel('Estratégias de Prompt Engineering', fontsize=12)

        plt.tight_layout()
        return fig

    def plot_exact_match_comparison(self):
        """Plota comparação de Exact Match"""
        fig, ax = plt.subplots(figsize=(12, 8))

        strategies = self.exact_match.index
        em_values = self.exact_match['SLM Base'].values

        # Criar gráfico de barras
        colors = plt.cm.viridis(np.linspace(0, 1, len(strategies)))
        bars = ax.bar(strategies, em_values, color=colors, alpha=0.8, edgecolor='black')

        ax.set_title('Comparação de Exact Match por Estratégia', fontsize=16, pad=20)
        ax.set_ylabel('Exact Match (%)', fontsize=12)
        ax.set_xlabel('Estratégias de Prompt Engineering', fontsize=12)
        ax.grid(True, alpha=0.3, axis='y')

        # Rotacionar labels
        plt.setp(ax.get_xticklabels(), rotation=45, ha='right')

        # Adicionar valores nas barras
        for bar, val in zip(bars, em_values):
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height + 0.1,
                   f'{val:.1f}%', ha='center', va='bottom', fontweight='bold')

        plt.tight_layout()
        return fig

    def plot_dataset_statistics(self):
        """Plota estatísticas do dataset"""
        fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))

        # Tamanhos dos conjuntos
        sizes = [self.dataset_stats['train_size'], self.dataset_stats['validation_size']]
        labels = ['Treino', 'Validação']
        colors = ['lightblue', 'lightcoral']

        ax1.pie(sizes, labels=labels, autopct='%1.1f%%', colors=colors, startangle=90)
        ax1.set_title('Distribuição dos Conjuntos de Dados', fontsize=14)

        # Estatísticas de comprimento
        stats_data = {
            'Contexto': {
                'Média': self.dataset_stats['avg_context_length'],
                'Desvio Padrão': self.dataset_stats['std_context_length']
            },
            'Pergunta': {
                'Média': self.dataset_stats['avg_question_length'],
                'Desvio Padrão': self.dataset_stats['std_question_length']
            }
        }

        categories = list(stats_data.keys())
        means = [stats_data[cat]['Média'] for cat in categories]
        stds = [stats_data[cat]['Desvio Padrão'] for cat in categories]

        x = np.arange(len(categories))
        ax2.bar(x, means, yerr=stds, capsize=5, color=['skyblue', 'lightgreen'],
                alpha=0.8, edgecolor='black')
        ax2.set_title('Estatísticas de Comprimento (Média ± DP)', fontsize=14)
        ax2.set_ylabel('Número de Palavras', fontsize=12)
        ax2.set_xticks(x)
        ax2.set_xticklabels(categories)
        ax2.grid(True, alpha=0.3)

        # Adicionar valores
        for i, (mean, std) in enumerate(zip(means, stds)):
            ax2.text(i, mean + std + 2, f'{mean:.1f}±{std:.1f}',
                    ha='center', va='bottom', fontweight='bold')

        # Gráfico de ranking de estratégias
        strategy_ranking = self.f1_scores['SLM Base'].sort_values(ascending=True)
        y_pos = np.arange(len(strategy_ranking))

        colors_rank = plt.cm.RdYlGn(np.linspace(0.2, 0.8, len(strategy_ranking)))
        ax3.barh(y_pos, strategy_ranking.values, color=colors_rank, alpha=0.8)
        ax3.set_yticks(y_pos)
        ax3.set_yticklabels(strategy_ranking.index)
        ax3.set_xlabel('F1-Score', fontsize=12)
        ax3.set_title('Ranking de Estratégias por Performance', fontsize=14)
        ax3.grid(True, alpha=0.3)

        # Adicionar valores
        for i, val in enumerate(strategy_ranking.values):
            ax3.text(val + 0.3, i, f'{val:.2f}', va='center', fontweight='bold')

        # Melhoria relativa ao baseline (Zero-Shot)
        baseline = self.f1_scores.loc['Zero-Shot', 'SLM Base']
        improvements = self.f1_scores['SLM Base'] - baseline
        improvements = improvements.drop('Zero-Shot')  # Remove o baseline

        colors_imp = ['green' if x > 0 else 'red' for x in improvements.values]
        ax4.bar(range(len(improvements)), improvements.values,
                color=colors_imp, alpha=0.7, edgecolor='black')
        ax4.set_title('Melhoria Relativa ao Zero-Shot', fontsize=14)
        ax4.set_ylabel('Diferença F1-Score', fontsize=12)
        ax4.set_xticks(range(len(improvements)))
        ax4.set_xticklabels(improvements.index, rotation=45, ha='right')
        ax4.axhline(y=0, color='black', linestyle='-', alpha=0.8)
        ax4.grid(True, alpha=0.3)

        # Adicionar valores
        for i, val in enumerate(improvements.values):
            color = 'white' if abs(val) > 5 else 'black'
            ax4.text(i, val + (0.5 if val > 0 else -0.5), f'{val:+.2f}',
                    ha='center', va='center' if abs(val) > 5 else ('bottom' if val > 0 else 'top'),
                    fontweight='bold', color=color)

        plt.tight_layout()
        return fig

    def generate_all_plots(self, save_plots=True, show_plots=True):
        """Gera todos os plots"""
        plots = {}

        print("Gerando visualizações dos resultados do experimento SLM...")

        # Gerar todos os plots
        plots['f1_distribution'] = self.plot_f1_distribution()
        plots['performance_comparison'] = self.plot_performance_comparison()
        plots['heatmap_performance'] = self.plot_heatmap_performance()
        plots['exact_match_comparison'] = self.plot_exact_match_comparison()
        plots['dataset_statistics'] = self.plot_dataset_statistics()

        # Salvar plots se solicitado
        if save_plots:
            for name, fig in plots.items():
                fig.savefig(self.figures_dir / f"{name}.png", dpi=300, bbox_inches='tight')
                fig.savefig(self.figures_dir / f"{name}.pdf", bbox_inches='tight')

            print(f"Plots salvos em: {self.figures_dir}")

        # Mostrar plots se solicitado
        if show_plots:
            plt.show()

        return plots

    def print_summary(self):
        """Imprime um resumo dos resultados"""
        print("\n" + "="*60)
        print("RESUMO DOS RESULTADOS DO EXPERIMENTO")
        print("="*60)

        print(f"\n📊 ESTATÍSTICAS DO DATASET:")
        print(f"   • Conjunto de treino: {self.dataset_stats['train_size']:,} exemplos")
        print(f"   • Conjunto de validação: {self.dataset_stats['validation_size']:,} exemplos")
        print(f"   • Comprimento médio do contexto: {self.dataset_stats['avg_context_length']:.1f} ± {self.dataset_stats['std_context_length']:.1f} palavras")
        print(f"   • Comprimento médio da pergunta: {self.dataset_stats['avg_question_length']:.1f} ± {self.dataset_stats['std_question_length']:.1f} palavras")

        print(f"\n🏆 MELHORES RESULTADOS:")
        best_f1 = self.f1_scores.max().max()
        best_combination = self.f1_scores.stack().idxmax()
        best_em = self.exact_match.max().max()

        print(f"   • Melhor F1-Score: {best_f1:.2f}")
        print(f"   • Melhor combinação: {best_combination[1]} + {best_combination[0]}")
        print(f"   • Melhor Exact Match: {best_em:.1f}%")

        print(f"\n📈 RANKING DE ESTRATÉGIAS (F1-Score):")
        ranking = self.f1_scores['SLM Base'].sort_values(ascending=False)
        for i, (strategy, score) in enumerate(ranking.items(), 1):
            print(f"   {i}. {strategy}: {score:.2f}")

        print("\n" + "="*60)

# Exemplo de uso
if __name__ == "__main__":
    # Inicializar visualizador
    # Os arquivos devem estar em: scientific_results/reports/
    # As figuras serão salvas em: scientific_results/figures/
    visualizer = SLMResultsVisualizer("scientific_results")

    # Imprimir resumo
    visualizer.print_summary()

    # Gerar todos os plots
    plots = visualizer.generate_all_plots(save_plots=True, show_plots=True)

    print("\n✅ Visualizações geradas com sucesso!")
    print(f"📁 Figuras salvas em: {visualizer.figures_dir}")
    print(f"📊 Dados carregados de: {visualizer.reports_dir}")

    # Lista de arquivos gerados
    generated_files = list(visualizer.figures_dir.glob("*.png"))
    if generated_files:
        print(f"\n📈 Arquivos gerados:")
        for file in sorted(generated_files):
            print(f"   • {file.name}")

    print("\n💡 Para usar novamente:")
    print("   visualizer = SLMResultsVisualizer('scientific_results')")
    print("   plots = visualizer.generate_all_plots()")